In [ ]:
#!/usr/bin/env python3


# plife-VCC : the simulation is the operator, the network rewrites its rules

## The architecture, and why each constraint is there

```
  perturbation p  ->  HYPERNETWORK  ->  RULES  Theta_p        (deep learning)
  x0 = injection at gene p
  x <- PLIFE(x ; Theta_p)   iterated T steps                  (the simulation)
  prediction = x                                              (identity readout)
```

**The plife simulation is the sole operator on gene expression.** The network
never emits an expression value. It emits the *rules* the simulation runs
under. That is the only division that survives the overfitting problem: the
rules are a few hundred numbers, the expression profile is 18,080.

### Three constraints that are not stylistic

1. **The operator must be NONLINEAR IN THE STATE.** This project already
   proved that linear dynamics plus a free readout is absorbable -- the
   composed map collapses to a single linear operator and propagation
   contributes exactly nothing (measured effect interval
   `[-0.00062, +0.00007]`). The nonlinearity is inside the update, not on the
   output.
2. **The readout is the IDENTITY.** A learned readout re-absorbs the
   simulation for the same reason. The state *is* the prediction.
3. **No free per-gene parameters.** Genes are described by fixed embeddings
   (DepMap co-essentiality, STRING, corpus co-response). A free per-gene table
   cannot generalise to genes never seen in the query role -- the cold-start
   failure that killed earlier versions of this model.

### Where the data comes from

The corpus is 22 contexts / 16,667 rows / 2.65M cells, but the honest count is
**7,585 UNIQUE perturbed genes**, not 16,667:

    magpie_hipsc           6,673 rows    6,664 unique     6,664 new
    magpie_line x19        8,204 rows      444 unique        34 new  <-- !
    joung_tfatlas_hesc     1,399 rows    1,399 unique       887 new
    joung_tf_diff            391 rows      391 unique         0 new

The 19 `magpie_line__*` files are ONE 444-gene core panel measured in 19
clones of 10 DONORS -- 18x replicates, not new perturbations -- and 92% of
that panel is already inside magpie_hipsc. `joung_tf_diff` adds nothing at all.

So: 7,585 unique targets, ~50x the 150 H1 perturbations, not the 111x a naive
row count suggests. Still the whole reason this is trainable, but stated
correctly. Three consequences are handled below:
  1. replicates are DOWNWEIGHTED, or 444 genes carry 19x the gradient mass
     purely because of how the study was sliced (circe2._study guards the same
     trap: "magpie is sliced ~20 ways and would otherwise outvote everything
     else for its core-panel genes")
  2. holdout is by DONOR, not by file -- holding out eipl_1 while eipl_3 trains
     is not a held-out context
  3. the replicates are still valuable: 19 clones on one panel is a direct
     measurement of how much a response moves across genetic backgrounds,
     which is exactly the H1-transfer question

Corpus files are pseudobulk (`delta` is already a per-perturbation mean) with
`n_cells` alongside. For learning a MEAN response, N cells sharing a mean is
mathematically identical to that mean weighted by N -- so `delta` weighted by
`n_cells` *is* per-cell training for first moments. Per-cell would only add
information about within-perturbation spread, which affects the Wilcoxon DE
calls, not MAE or PDS.

### H1 as the dictatorial set

Rules are learned across all pluripotent contexts. H1 never votes on the
rules. It enters twice, both times as a constraint on an already-learned
operator:
  * as a context vector (its own control-cell baseline) fed to the hypernet
  * as a final low-rank calibration fitted on the 150 H1 perturbations
    -- "take these relationships and make them fit this well condition"

### Protocol

  * rules trained on non-H1 pluripotent contexts only
  * model selection by LEAVE-CONTEXT-OUT (hold out whole donor lines) --
    the honest test of whether rules transfer to an unseen context
  * H1 calibration fitted on the 150 training perturbations under fold-out
  * the 50 validation perturbations are scored ONCE, at the end
  * `predict ZERO` and `PERFECT` printed alongside every result


## PDS-constrained revision

This version keeps plife as the predictor, but changes the training signal and
selection rule:

- adds a deterministic, same-context pairwise PDS margin loss using the scorer's anchor-target rule;
- computes PDS and shape losses on perturbations from ONE context at a time,
  matching the actual VCC comparison;
- ramps PDS to full strength by 10% of training, before any lifecycle pruning;
- keeps dynamic sparsity and shape loss off for the first stable-PDS test;
- separates the target column from off-target calibration;
- freezes `de_score` to the H1 DE-frequency prior so DES cannot regress;
- selects `alpha` by highest H1-training PDS subject to MAE/DES constraints;
- separates lifecycle switches so they can be audited independently.

Default first run: connectivity dropout is on for the pointwise fit, but the PDS pass is deterministic; all other lifecycle mechanisms are off.


In [ ]:
import os, math, glob, time, json
import numpy as np

# ---------------------------------------------------------------- data path --
# AUTO-DISCOVERED. The dataset slug and folder depth vary with how it was
# uploaded (e.g. /kaggle/input/datasets/<user>/<slug>/), and Kaggle does not
# preserve subfolders reliably, so nothing here is hardcoded: find the directory
# that actually contains vcc_pseudobulk.npz and work from there. Files may be
# flat or in a corpus/ subfolder -- both work.
INPUT_DIR = None
for _root, _d, _f in os.walk("/kaggle/input"):
    if "vcc_pseudobulk.npz" in _f:
        INPUT_DIR = _root
        break
if INPUT_DIR is None:                       # local / non-Kaggle fallback
    INPUT_DIR = os.environ.get("PLIFE_INPUT", ".")
CORPUS_DIR = os.path.join(INPUT_DIR, "corpus")
print("INPUT_DIR :", INPUT_DIR)
print("files here:", len([x for x in glob.glob(os.path.join(INPUT_DIR, '*.npz'))]),
      "npz at root,",
      len(glob.glob(os.path.join(CORPUS_DIR, '*.npz'))), "in corpus/")

EPOCHS       = 1200        # compute is not the constraint here
M_MODES      = 96          # plife field dimension
T_STEPS      = 6           # simulation steps
HID          = 1024        # LOCKED. From the 13-config sweep (acc mean +- sd):
#     2x1024  +0.0110 +- 0.0078      3x1024  +0.0126 +- 0.0089   <- best
#     3x384   +0.0106 +- 0.0067      3x512   +0.0078 +- 0.0053
#     everything at depth 4+        <= 0.000
# Depth 2-3 is the usable region and deeper is strictly worse. The differences
# inside that region are within one sd of each other, so this is "pick the best
# of the indistinguishable good ones", not a claim that 3x1024 is special.
DEPTH        = 3           # LOCKED with HID above
DENSITY      = 0.10        # final sparsity after pruning
DROPOUT      = 0.15        # MEAN rate, held fixed regardless of spread
DROP_SPREAD  = 0.8         # 0 = uniform dropout; 1 = rate spans [0, 2*DROPOUT]
LR           = 2e-3
BATCH        = 512
SEED         = 0
HOLDOUT_CTX  = 4           # donor lines held out for leave-context-out
GATE_TEMP    = 1.0         # gene-competition temperature; low = winner-take-all
# --- constrained-PDS revision switches --------------------------------------
USE_PDS_LOSS = True        # train the pairwise ranking PDS actually measures
PDS_W        = 0.25        # margin ranking weight; isolate before scaling up
PDS_B        = 64          # exact anchor x candidate subset; 512 x 512 x G is wasteful
PDS_WARMUP   = 0.0         # start immediately; ramp controls how fast
PDS_RAMP     = 0.10        # full PDS weight by 10% of training
PDS_LOSS     = "margin"    # "margin" is stabler than listwise CE
USE_SHAPE_LOSS = True      # match real response top-k L1 mass
SHAPE_W      = 0.0         # OFF for the first stable-PDS test
USE_DYNAMIC_SPARSITY = False # OFF for the first stable-PDS test
TAU_MAX      = 0.10

# Lifecycle controls, separated so each mechanism can be credited or blamed.
USE_CONN_DROPOUT = True
USE_PRUNE        = False     # OFF until context-homogeneous PDS learns
USE_REGROW       = False     # OFF with pruning
USE_STREAMLINE   = False      # first audit pass: OFF until the static baseline is known
USE_UNIT_DEATH   = False      # first audit pass: OFF
USE_GENE_DEATH   = False      # hard gene death previously hurt PDS
STREAMLINE_WARMUP = 0.25
LIFECYCLE_AUDIT  = True
LIFECYCLE_B      = 64
SCORE_VALIDATION = False      # keep False for ablations; set True only for the final run
MAX_DEAD         = 0.20       # cap if USE_GENE_DEATH is enabled later
#
# MEASURED, before you trust hard death: applied post-hoc to the current best
# retrieval prediction, killing quiet genes made the score WORSE, monotonically.
#
#     current best (adaptive dropout)          40.00
#       + kill 4,208 quiet genes               39.74
#       + kill quietest 10% / 25% / 40%        39.82 / 39.74 / 39.31
#
# MAE did not move at all (0.0112 either way) -- the retrieval predictor already
# predicts ~0 on quiet genes, so there was no wasted L1 mass to reclaim, and PDS
# fell, meaning those genes carried a little discriminative signal.
#
# That is a fact about a well-calibrated predictor, NOT about this network. A
# net can waste capacity and put spurious mass on genes that never move, which
# is what the gate is for. The SOFT gate (continuous, learned, competitive) is
# the part likely to help; hard death is the blunt version that measurably hurt.
# So: soft gate always on, hard death capped and ablatable. Run it both ways.

import torch
import torch.nn as nn
import torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", dev, "| torch", torch.__version__)


## 1. Load every pluripotent context


In [ ]:
def find_corpus():
    """Locate the corpus files WHEREVER they landed.

    Kaggle frequently flattens an uploaded directory tree, so `corpus/` may not
    survive as a subfolder -- the 22 files then sit at the dataset root next to
    vcc_pseudobulk.npz. Searching recursively costs nothing and removes an
    entire class of silent failure.
    """
    seen, out = set(), []
    for pat in (os.path.join(CORPUS_DIR, "*.npz"),
                os.path.join(INPUT_DIR, "*.npz"),
                os.path.join(INPUT_DIR, "**", "*.npz"),
                "/kaggle/input/**/*.npz"):
        for f in glob.glob(pat, recursive=True):
            b = os.path.basename(f)
            if b in seen:
                continue
            if any(k in b.lower() for k in ("magpie", "joung")):
                seen.add(b); out.append(f)
    return sorted(out)


def load_contexts():
    files = find_corpus()
    ctxs = []
    for f in files:
        z = np.load(f, allow_pickle=True)
        name = str(z["context"][0])
        low = name.lower()
        if not any(k in low for k in ("magpie", "joung")):
            continue                                   # pluripotent only, H1 excluded
        # donor identity: magpie_line::eipl_1 and ::eipl_3 are CLONES of one
        # donor and must be held out together
        if "magpie_line" in low:
            donor = name.split("::")[1].rsplit("_", 1)[0]
        elif "magpie" in low:
            donor = "magpie_pool"
        else:
            donor = name
        ctxs.append(dict(name=name, donor=donor,
                         perts=[str(p).split("_")[0] for p in z["perts"]],
                         delta=z["delta"].astype(np.float32),
                         n=z["n_cells"].astype(np.float64),
                         base=z["baseline"].astype(np.float32),
                         meas=z["measured"]))
    return ctxs

ctxs = load_contexts()
zc = np.load(os.path.join(INPUT_DIR, "vcc_pseudobulk.npz"), allow_pickle=True)
GENES = [str(g) for g in zc["genes"]]; GPOS = {g: i for i, g in enumerate(GENES)}
G = len(GENES)
H1_lfc  = zc["lfc"].astype(np.float32)
H1_pert = [str(p) for p in zc["perts"]]
H1_n    = zc["cnts"].astype(np.float64)
H1_base = zc["ctrl"].astype(np.float32)
print(f"{len(ctxs)} non-H1 pluripotent contexts, "
      f"{sum(len(c['perts']) for c in ctxs):,} perturbations, "
      f"{sum(c['n'].sum() for c in ctxs):,.0f} cells")

# FAIL HERE, LOUDLY. With an empty corpus every downstream cell still "runs" --
# the first run died four cells later on `max() arg is empty`, which says
# nothing about the real problem.
if len(ctxs) == 0:
    print("\n  NO CORPUS FILES FOUND. What is actually under INPUT_DIR:")
    for root, dirs, fs in os.walk(INPUT_DIR):
        lvl = root.replace(INPUT_DIR, "").count(os.sep)
        print("   " + "  " * lvl + os.path.basename(root) + "/")
        for x in sorted(fs)[:12]:
            print("   " + "  " * (lvl + 1) + x)
        if len(fs) > 12:
            print("   " + "  " * (lvl + 1) + f"... and {len(fs) - 12} more")
    raise SystemExit(
        f"Expected 22 files named magpie*/joung*.npz somewhere under "
        f"/kaggle/input (looked in {INPUT_DIR}, {CORPUS_DIR}, and recursively). "
        "Found none -- they are missing from the dataset. Flat or in a "
        "corpus/ subfolder both work; the path is auto-discovered.")
assert len(ctxs) >= 20, (f"only {len(ctxs)} corpus contexts found, expected 22 "
                         "-- the upload is incomplete")
print(f"H1: {len(H1_pert)} training perturbations, {H1_n.sum():,.0f} cells")


## 2. Fixed gene descriptors — no free per-gene parameters

Every gene, perturbed or not, is described by the same fixed features, so a
gene never seen as a perturbation target is not cold.


In [ ]:
# STRING edges only -- 3 MB, extracted from plife_data.npz. The full
# plife_data.npz is 416 MB and 85% of it is corpus delta matrices this notebook
# reads from corpus/ instead, so uploading it would be dead weight.
_sf = os.path.join(INPUT_DIR, "string_edges.npz")
pl = np.load(_sf if os.path.isfile(_sf)
             else os.path.join(INPUT_DIR, "plife_data.npz"), allow_pickle=True)
_k = "i" if "i" in pl.files else "string_i"
Z = []
E64 = np.load(os.path.join(INPUT_DIR, "emb_depmap.npz"), allow_pickle=True)["E"].astype(np.float32)
Z.append(E64 / (np.linalg.norm(E64, axis=1, keepdims=True) + 1e-9))

si = pl[_k]; sj = pl["j" if _k == "i" else "string_j"]
sv = pl["score" if _k == "i" else "string_score"].astype(np.float32)
Sm = np.zeros((G, 256), np.float32)                    # STRING profile, projected
rp = np.random.default_rng(0).normal(0, 1, (G, 256)).astype(np.float32) / 16
np.add.at(Sm, si, rp[sj] * sv[:, None])
np.add.at(Sm, sj, rp[si] * sv[:, None])
Sm /= (np.linalg.norm(Sm, axis=1, keepdims=True) + 1e-9)
Z.append(Sm)

lb = np.log1p(np.maximum(H1_base, 0)); Z.append((lb / (lb.std() + 1e-9))[:, None])
ZG = torch.tensor(np.concatenate(Z, 1), dtype=torch.float32, device=dev)
D_GENE = ZG.shape[1]
print("gene descriptor:", tuple(ZG.shape))


## 3. The plife operator

Genes are particles. Their positions come from the fixed descriptors, so the
geometry is given, not learned. What the network changes is the FORCE LAW.

```
  u   = P^T x                     the field: genes push into m shared modes
  f_g = P_g . (A u)               force on gene g, A = the interaction rules
  x  <- x + dt ( -gamma x + tanh(f_g + b_g) )
```

`tanh` inside the update is what makes the composition non-absorbable. `A` is
built as `U diag(s) V^T` with `U, V` global and **only the spectrum `s`
emitted per perturbation** -- so the hypernetwork rewrites the strength of
each interaction mode without ever touching a gene directly.


In [ ]:
class GeneViability(nn.Module):
    """Genes COMPETE for a fixed participation budget, and the losers die.

    Measured on the pluripotent corpus (8,072 perturbations):

        rank corr(corpus activity, H1 activity)      +0.8154
        bottom 25% by corpus activity (4,520 genes)  2.92% of H1 response mass
        corpus-quiet vs active genes, H1 activity    11.1x less

    A gene that never moves anywhere carries no signal, and because MAE and PDS
    are L1 over all 18,080 genes, predicting EXACTLY ZERO for it is optimal.
    Killing it is not merely regularisation -- it removes L1 mass that can only
    ever be wrong.

    Softmax rather than independent thresholds, so this is genuinely
    competitive: the budget is fixed, `a` averages 1 by construction, and a gene
    can only gain participation at another gene's expense. That is what makes it
    a selection process instead of 18,080 unrelated gates drifting toward zero.

        logit_g = score(z_g) + log(corpus activity_g)
        a       = G * softmax(logit / temperature)

    `a_g` scales the gene's row in the projection -- so a dying gene stops
    pushing into the field AND stops receiving force -- and scales its output,
    so death drives its prediction to exactly zero.

    DEATH is permanent, which makes it dangerous, so it is guarded:
      * nothing dies before `warmup`; early gate values are noise
      * a gene must stay below threshold for `patience` consecutive checks,
        judged on an EMA, not a single noisy reading
      * at most `max_dead` of all genes may ever die, so the layer cannot
        collapse to a handful of survivors
      * the corpus activity prior comes from NON-H1 contexts only
    """

    def __init__(self, d_gene, G, act_prior, temp=1.0, max_dead=0.45,
                 patience=3, warmup=0.25, protect=None):
        super().__init__()
        self.G, self.temp = G, temp
        self.max_dead, self.patience, self.warmup = max_dead, patience, warmup
        self.score = nn.Sequential(nn.Linear(d_gene, 64), nn.GELU(), nn.Linear(64, 1))
        self.register_buffer("prior", torch.log(act_prior + 1e-6))
        self.register_buffer("alive", torch.ones(G))
        # Genes that are PERTURBATION TARGETS are immortal. If a target dies,
        # its projection row is zeroed, the injected field is exactly zero, and
        # the whole simulation for that perturbation collapses -- which is also
        # where the NaN came from. A gene we are asked to predict the effect of
        # cannot be one the model has decided is irrelevant.
        pr = torch.zeros(G)
        if protect is not None:
            pr[torch.as_tensor(sorted(protect), dtype=torch.long)] = 1.0
        self.register_buffer("immortal", pr)
        self.register_buffer("ema", torch.ones(G))
        self.register_buffer("strikes", torch.zeros(G))

    def gate(self, ZG):
        logit = self.score(ZG).squeeze(-1) + self.prior
        a = self.G * torch.softmax(logit / self.temp, dim=0)
        return a * self.alive

    @torch.no_grad()
    def reap(self, ZG, t, thresh=0.05):
        """One death check. Returns number newly dead."""
        a = self.gate(ZG)
        self.ema.mul_(0.8).add_(0.2 * a)
        if t < self.warmup:
            return 0
        low = (self.ema < thresh) & (self.alive > 0) & (self.immortal == 0)
        self.strikes[low] += 1
        self.strikes[~low] = 0
        cand = (self.strikes >= self.patience) & (self.alive > 0)
        budget = int(self.max_dead * self.G) - int((self.alive == 0).sum())
        if budget <= 0 or not bool(cand.any()):
            return 0
        idx = torch.nonzero(cand).squeeze(-1)
        if len(idx) > budget:                       # kill the quietest first
            idx = idx[torch.argsort(self.ema[idx])[:budget]]
        self.alive[idx] = 0.0
        return len(idx)

    def n_dead(self):
        return int((self.alive == 0).sum())


class Plife(nn.Module):
    def __init__(self, d_gene, m, steps):
        super().__init__()
        self.m, self.steps = m, steps
        self._sat = 0.0
        self._drive_zero = 0.0
        self.proj = nn.Sequential(nn.Linear(d_gene, 256), nn.GELU(), nn.Linear(256, m))
        self.U = nn.Parameter(torch.randn(m, m) / math.sqrt(m))
        self.V = nn.Parameter(torch.randn(m, m) / math.sqrt(m))
        # The interaction scale is NORMALISED, not guessed. Both previous runs
        # died on this constant from opposite directions:
        #   coupling too small -> |f| ~ 0.003, tanh(f+b) = the bias, no
        #                          propagation, off-target output ~0
        #   coupling = 3.0     -> |f| ~ 250,   tanh 99.4% SATURATED, gradient ~0,
        #                          nothing learns (train cosine ~0 too)
        # Measured saturation: 0.0% at 0.1, 95.4% at 1.0, 99.4% at 3.0.
        # So the force is RMS-normalised per batch before tanh and a learned
        # `gain` sets the operating point on the nonlinearity. The model can now
        # choose how nonlinear to be instead of inheriting it from an init.
        self.gain_f = nn.Parameter(torch.tensor(0.5))
        self.bias = nn.Sequential(nn.Linear(d_gene, 64), nn.GELU(), nn.Linear(64, 1))

    def forward(self, x0, s, gamma, dt, gain, ZG, a=None, tau=None):
        P = self.proj(ZG)                                  # (G, m) fixed geometry
        P = P / (P.norm(dim=1, keepdim=True) + 1e-9)       # unit rows
        b = self.bias(ZG).squeeze(-1)                      # (G,)
        if a is not None:
            P = P * a[:, None]        # a dying gene neither pushes nor receives
        x = x0
        for _ in range(self.steps):
            u = x @ P                                      # (B, m) the field
            Au = (u @ self.V) * s                          # rules modulate the spectrum
            Au = Au @ self.U.T
            f = Au @ P.T                                   # (B, G) force per gene
            # RMS-normalise so tanh operates in its responsive band regardless
            # of how large the learned projections happen to be
            # NO SQRT. `x.sqrt()` is singular at x = 0 and f IS exactly zero
            # whenever the perturbed gene has been gated off by GeneViability:
            # P[target] = 0 -> u = 0 -> f = 0. Finite-difference on the shipped
            # version gave d(out)/df = 9.1e5 at f = 0, which is the NaN that
            # returned from all 39 sweep configs at 451s each. Mean-ABSOLUTE
            # normalisation has no singularity anywhere and suits an L1 metric.
            f = f / f.abs().mean(1, keepdim=True).clamp_min(1e-4)
            z = self.gain_f * f + b[None, :]
            if tau is not None:
                # Perturbation-specific soft-threshold: exact zeros for weak drives,
                # unlike the global gene gate, which cannot create per-row sparsity.
                z = torch.sign(z) * torch.relu(z.abs() - tau[:, None])
            drive = torch.tanh(z)
            self._sat = float((drive.abs() > 0.99).float().mean())      # WATCH THIS
            self._drive_zero = float((drive == 0).float().mean())
            x = x + dt[:, None] * (-gamma[:, None] * x + drive)
        if a is not None:
            x = x * a[None, :]        # dead gene -> exactly zero, optimal under L1
        return x * gain[:, None]                           # identity readout, scaled


class CrossLayerHyper(nn.Module):
    """Over-parameterised, CONNECTIVITY-WEIGHTED dropout, magnitude pruning,
    cross-layer synaptogenesis. Emits ONLY rules: mode spectrum, decay,
    timestep and gain.

    ### Connectivity-weighted dropout

    Uniform dropout treats a unit holding 400 surviving synapses and a unit
    holding 3 identically. Under prune-and-regrow the connectivity is changing
    every cycle, so it is a live signal about which pathways the network has
    actually confirmed. Here the drop probability RISES as a unit loses
    connections:

        c_j        surviving in-edges + out-edges of unit j (all cross-layer
                   matrices included, so a unit fed by an L1->L3 skip counts it)
        r_j        c_j rank-normalised to [0, 1] within its layer
        p_j        p_drop * (1 + SPREAD * (1 - 2 r_j))

    The MEAN of p_j is exactly p_drop, by construction, for any SPREAD. That
    matters: the obvious parameterisation `p_lo + (p_hi-p_lo)(1-r)` centres the
    rate at (p_lo+p_hi)/2, so switching the connectivity signal on also raises
    the average drop rate -- measured, 0.150 -> 0.225 -- and any change in the
    result would confound "dropout weighted by connectivity" with "simply more
    dropout". Here only the DISTRIBUTION of dropout moves; the total amount is
    held fixed, so the comparison against uniform dropout is controlled.

    Well-connected units are rarely silenced and keep receiving gradient;
    sparsely-connected ones are silenced often, so the network is pushed to
    consolidate around pathways that already survived pruning.

    Three guards, because this is a rich-get-richer loop and left alone it
    collapses to a narrow subnetwork:

      * p is FLOORED at p_lo and CAPPED at p_hi -- no unit is ever undroppable
        (which would let it memorise) and none is ever certain to drop (which
        would kill it permanently rather than regularise it)
      * the coupling is ANNEALED IN. Early training is ordinary uniform dropout,
        because connectivity is meaningless before any pruning has happened;
        the signal only takes over once it carries information
      * inverted scaling is PER UNIT, 1/(1 - p_j). With a shared p the layer's
        expected activation would drift as the connectivity distribution moves,
        silently rescaling everything downstream
    """

    def __init__(self, d_in, hid, depth, m, p_drop, spread=0.8):
        super().__init__()
        self.depth = depth
        self.p_drop = p_drop
        self.spread = spread          # 0 = uniform, 1 = rate spans [0, 2*p_drop]
        self.coupling = 0.0                      # annealed 0 -> 1 during training
        dims = [d_in] + [hid] * depth
        self.pairs = [(a, b) for a in range(depth) for b in range(a + 1, depth + 1)]
        self.lin = nn.ModuleList([nn.Linear(dims[a], hid, bias=(b == a + 1))
                                  for (a, b) in self.pairs])
        self.register_buffer("_ones", torch.ones(1))
        self.masks = [torch.ones_like(l.weight) for l in self.lin]
        self.head = nn.Linear(hid, m + 4)
        self.hid = hid
        self._pdrop = [None] * (depth + 1)
        self._pre = None
        self.tiers = [0, 0, 0]
        self._idx = {pr: i for i, pr in enumerate(self.pairs)}   # (src,dst) -> matrix
        w0 = self.lin[0].weight
        self.unit_alive = [None] + [torch.ones(hid, device=w0.device)
                                    for _ in range(depth)]
        self.vit_ema = [None] + [torch.ones(hid, device=w0.device)
                                 for _ in range(depth)]
        self.refresh_connectivity()

    def to_dev(self, d):
        self.masks = [m.to(d) for m in self.masks]
        self.unit_alive = [None] + [x.to(d) for x in self.unit_alive[1:]]
        self.vit_ema = [None] + [x.to(d) for x in self.vit_ema[1:]]
        self.refresh_connectivity()
        return self

    def n_units_dead(self):
        return int(sum(float((x == 0).sum()) for x in self.unit_alive[1:]))

    @torch.no_grad()
    def refresh_connectivity(self):
        """Recount surviving edges per unit. Call after every prune/grow --
        connectivity is what the dropout probability is conditioned on, so it
        must not go stale."""
        dev_ = self.masks[0].device
        for l in range(1, self.depth + 1):
            inc = torch.zeros(self.hid, device=dev_)
            out = torch.zeros(self.hid, device=dev_)
            for i, (a, b) in enumerate(self.pairs):
                if b == l:
                    inc += self.masks[i].sum(1)          # (hid,) in-edges
                if a == l:
                    out += self.masks[i].sum(0)          # (hid,) out-edges
            if l == self.depth:
                out = out + float(self.head.weight.shape[0])
            c = inc + out
            # rank-normalise within the layer: robust to the overall density
            # collapsing during the pruning schedule
            r = torch.argsort(torch.argsort(c)).float() / max(len(c) - 1, 1)
            # mean is exactly p_drop for any spread -- only the DISTRIBUTION of
            # dropout depends on connectivity, never the total amount
            self._pdrop[l] = self.p_drop * (1.0 + self.spread * (1.0 - 2.0 * r))

    def _conn_dropout(self, h, l):
        if (not self.training) or (not USE_CONN_DROPOUT):
            return h
        p_uniform = torch.full_like(self._pdrop[l], self.p_drop)
        p = (1 - self.coupling) * p_uniform + self.coupling * self._pdrop[l]
        p = p.clamp(0.0, 0.95)
        keep = (torch.rand(h.shape, device=h.device) >= p[None, :]).float()
        return h * keep / (1.0 - p)[None, :]              # per-unit inverted scaling

    # ------------------------------------------------------------------ death
    @torch.no_grad()
    def unit_vitality(self):
        """MEAN OUTGOING MAGNITUDE over surviving synapses -- sign-agnostic.

        The previous version scored `positive_mass + 0.5 * |mass|`, which
        rewarded units whose outgoing weights happened to be positive. That is
        wrong for this model: an inhibitory pathway is as real as an excitatory
        one, and a large negative weight drives the simulation just as hard.
        Vitality is now the weighted MEAN of |w| over live edges, so a unit with
        few strong synapses is not penalised against one with many weak ones.
        """
        out = []
        dv = self.masks[0].device
        for l in range(1, self.depth + 1):
            tot = torch.zeros(self.hid, device=dv)
            cnt = torch.zeros(self.hid, device=dv)
            for i, (a, b) in enumerate(self.pairs):
                if a == l:
                    w = self.lin[i].weight * self.masks[i]      # (hid_dst, hid_src)
                    tot += w.abs().sum(0)
                    cnt += self.masks[i].sum(0)
            if l == self.depth:
                tot += self.head.weight.abs().sum(0)
                cnt += float(self.head.weight.shape[0])
            out.append(tot / cnt.clamp(min=1.0))                # MEAN magnitude
        return out

    @torch.no_grad()
    def reap_units(self, t, warmup=0.35, p_max=0.02, p_bg=0.0002, offset=5.0,
                   min_alive=0.75):
        """DEATH IS TERMINAL, NOT COMPETITIVE. Dropout is the universal
        background; death is reserved for units that are already functionally
        disconnected.

        Why this and not a vitality hazard: death is irreversible, so making it
        a bet on a noisy signal is the expensive kind of mistake. A unit with no
        surviving outgoing synapses contributes *exactly nothing* downstream --
        collecting it changes the network's output by zero. That makes death
        function-preserving garbage collection rather than a gamble, and it puts
        the mechanisms in one causal chain instead of two competing ones:

            connectivity-weighted dropout silences weak units more
              -> they receive less gradient
              -> their weights decay
              -> magnitude pruning severs their synapses
              -> death certifies what has already happened

        MEASURED, and it changes the rule twice. First, STRICT disconnection
        essentially never happens. Simulating this trunk at 512 wide, depth 3:

            density   units reaching zero in- or out-edges
              0.30            0 / 1536
              0.10            0 / 1536      <- our operating point
              0.02            0 / 1536
              0.01           25 / 1536  (1.6%)

        So death-for-the-DISconnected is a no-op at any density we run at. The
        criterion is UNDERconnection, not disconnection.

        Second, a threshold relative to the median is fragile, because the
        connectivity distribution is tight -- measured spread 34-71 around a
        median of 51, so even a 0.7 x median cutoff caught 4 units of 512, and
        0.5 x caught none. A fixed cutoff would fire wildly differently on a
        skewed distribution than a tight one.

        So the hazard is SMOOTH and scaled by the spread itself:

            u_j = (median(conn) - conn_j) / MAD(conn)      how far below, in MADs
            h_j = p_bg + (p_max - p_bg) * sigmoid(u_j - 4)

        No percentile, no tier, no fixed fraction condemned. A unit at the
        median carries h ~ 0.0014 (about 95% survival over a run); one four MADs
        below carries 0.025 (~42%); one eight MADs below carries 0.049 (~18%).
        Being underconnected makes a unit *definitely more prone* to death
        without ever making it certain, and every unit keeps the small nonzero
        floor p_bg. If the distribution is tight, almost nothing dies -- which
        is the correct behaviour, because then nothing is underconnected.

        Literal zero in- or out-edges remains certain death: that unit already
        contributes exactly nothing, so collecting it is free.
        """
        if t < warmup:
            return 0
        n_dead = 0
        self.tiers = [0, 0, 0]   # zero-edge / underconnected / healthy -- WATCH THIS
        for l in range(1, self.depth + 1):
            alive = self.unit_alive[l]
            if float(alive.sum()) <= min_alive * self.hid:
                continue
            inc = torch.zeros(self.hid, device=alive.device)
            out = torch.zeros(self.hid, device=alive.device)
            for i, (a, b) in enumerate(self.pairs):
                if b == l:
                    inc += self.masks[i].sum(1)
                if a == l:
                    out += self.masks[i].sum(0)
            if l == self.depth:
                out = out + float(self.head.weight.shape[0])
            # signal = MEAN OUTGOING MAGNITUDE (sign-agnostic), not edge count.
            # Edge count says how many synapses a unit has; magnitude says how
            # hard it actually drives. A unit with 3 strong synapses is alive.
            conn = self.unit_vitality()[l - 1]
            liv = conn[alive > 0]
            med = liv.median()
            mad = (liv - med).abs().median() + 1e-6      # spread, robustly
            # how far BELOW the median, measured in MADs -- adapts to the actual
            # distribution instead of condemning a fixed fraction
            u = (med - conn) / mad
            haz = p_bg + (p_max - p_bg) * torch.sigmoid(u - offset)
            gone = ((inc == 0) | (out == 0)) & (alive > 0)   # truly severed only
            self.tiers[0] += int(gone.sum())
            self.tiers[1] += int(((u > offset) & (alive > 0)).sum())   # underconnected
            self.tiers[2] += int(((alive > 0) & ~gone & (u <= offset)).sum())
            die = ((torch.rand_like(haz) < haz) | gone) & (alive > 0)
            room = int(float(alive.sum()) - min_alive * self.hid)
            if int(die.sum()) > room:                    # least connected first
                idx = torch.nonzero(die).squeeze(-1)
                idx = idx[torch.argsort(conn[idx])[:max(room, 0)]]
                die = torch.zeros_like(die)
                die[idx] = True
            self.unit_alive[l][die] = 0.0
            for i, (a, b) in enumerate(self.pairs):      # sever its synapses
                if a == l:
                    self.masks[i][:, die] = 0.0
                if b == l:
                    self.masks[i][die, :] = 0.0
            n_dead += int(die.sum())
        if n_dead:
            self.refresh_connectivity()
        return n_dead

    # ------------------------------------------- cross-layer synaptogenesis
    @torch.no_grad()
    def streamline(self, frac=0.35, relay_q=0.35, per_layer=64, min_slope=0.05):
        """n1 -> n2 -> n3 where n2 is a PASS-THROUGH: build n1 -> n3 directly.

        The trigger is your invariance condition, made testable. The n1 -> n3
        contribution routed through n2 is `w32 * phi(w21 a1 + everything else)`.
        That contribution is independent of "everything else" exactly when phi
        is in its linear regime -- then it separates into
        `w32 * phi' * w21 * a1` plus terms that do not involve n1. So:

            n2 is a relay  <=>  phi'(pre_2) is near-CONSTANT across inputs
                            <=>  Var[phi'(pre_2)] is small

        which is cheap to measure. For relay units the composite gain is known
        in closed form:

            C[n3, n1] = w32[n3, n2] * mean(phi'_2) * w21[n2, n1]

        so the new direct edge is born AT THAT VALUE rather than at zero -- this
        is consolidating a confirmed pathway, not exploring a new one, and the
        value is already known. A matching fraction is removed from the n1 -> n2
        leg, so the function is approximately preserved while the path shortens
        and a synapse is freed.

        Approximate, not exact: n2 may feed units other than n3, and weakening
        its input changes those too. `frac` < 1 keeps the edit gentle and lets
        training re-converge.
        """
        if self._pre is None or self.depth < 2:
            return 0
        grown = 0
        for mid in range(1, self.depth):
            src_l, dst_l = mid - 1, mid + 1
            di = self._idx.get((src_l, dst_l))
            if di is None:
                continue
            pre = self._pre[mid]
            gp = torch.sigmoid(1.702 * pre) + 1.702 * pre * torch.sigmoid(1.702 * pre) \
                * (1 - torch.sigmoid(1.702 * pre))              # d/dx GELU (approx)
            gbar, gvar = gp.mean(0), gp.var(0)
            live = self.unit_alive[mid] > 0
            if int(live.sum()) < 4:
                continue
            thr = torch.quantile(gvar[live], relay_q)
            # Consistently linear AND responsive. A saturated/dead unit also has
            # low derivative variance, but it is not a useful relay.
            relay = live & (gvar <= thr) & (gbar >= min_slope)
            if int(relay.sum()) == 0:
                continue
            i_in = self._idx.get((src_l, mid))
            i_out = self._idx.get((mid, dst_l))
            if i_in is None or i_out is None:
                continue
            W_in = self.lin[i_in].weight * self.masks[i_in]     # (hid, d_src)
            W_out = self.lin[i_out].weight * self.masks[i_out]  # (hid, hid)
            r = torch.nonzero(relay).squeeze(-1)
            C = (W_out[:, r] * gbar[r][None, :]) @ W_in[r, :]   # (hid_dst, d_src)
            absent = (self.masks[di] == 0)
            cand = C.abs() * absent
            k = min(per_layer, int(absent.sum()))
            if k <= 0:
                continue
            flat = cand.flatten()
            thrv = torch.topk(flat, k).values.min()
            add = (cand >= thrv) & absent
            self.masks[di][add] = 1.0
            self.lin[di].weight[add] = (frac * C)[add]          # born AT the gain
            # transfer: weaken the leg being bypassed
            self.lin[i_in].weight[r, :] *= (1.0 - frac)
            grown += int(add.sum())
        if grown:
            self.refresh_connectivity()
        return grown

    def forward(self, e):
        acts = [e] + [None] * self.depth
        pres = [None] * (self.depth + 1)
        for l in range(1, self.depth + 1):
            tot = 0
            for i, (a, b) in enumerate(self.pairs):
                if b == l:
                    tot = tot + F.linear(acts[a], self.lin[i].weight * self.masks[i],
                                         self.lin[i].bias)
            pres[l] = tot
            acts[l] = self._conn_dropout(F.gelu(tot), l) * self.unit_alive[l][None, :]
        # Evaluation probes must not overwrite the training-batch relay statistics.
        if self.training:
            self._pre = [None] + [p.detach() for p in pres[1:]]
        o = self.head(acts[self.depth])
        m = o.shape[1] - 4
        # SPECTRUM MAY BE NEGATIVE. It was `1 + 0.5*tanh` = (0.5, 1.5): strictly
        # positive, so the hypernetwork could only scale each interaction mode
        # between 0.5x and 1.5x. It could never INVERT a mode (make an
        # activating interaction inhibitory for one perturbation) and never
        # switch one off. Since the spectrum is the ONLY thing that differs
        # between perturbations, that capped how distinguishable two
        # perturbations' operators could be. Measured, angle between A(p1) and
        # A(p2) over random draws:
        #     s in (0.5, 1.5)  CURRENT      22.6 deg   -- nearly the same operator
        #     s in (0, 2)      can switch off 41.5 deg
        #     s in (-2, 2)     can invert     90.2 deg  <- genuinely different
        # Biology needs the sign: knocking down a repressor RAISES its targets,
        # knocking down an activator LOWERS them. A strictly positive spectrum
        # cannot express both through the same mode.
        s = 2.0 * torch.tanh(o[:, :m])                     # spectrum in (-2, 2)
        gamma = 0.5 * torch.sigmoid(o[:, m])               # decay in (0, .5)
        dt = 0.5 * torch.sigmoid(o[:, m + 1])              # step in (0, .5)
        gain = F.softplus(o[:, m + 2]) + 1e-3
        tau = TAU_MAX * torch.sigmoid(o[:, m + 3])
        return s, gamma, dt, gain, tau

    def density(self):
        return float(sum(m.sum() for m in self.masks) /
                     sum(m.numel() for m in self.masks))

    @torch.no_grad()
    def prune_and_grow(self, dens, grow):
        flat = torch.cat([(l.weight * m).abs().flatten()
                          for l, m in zip(self.lin, self.masks)])
        keep = max(1, int(dens * flat.numel()))
        thr = torch.topk(flat, keep).values.min()
        for l, m in zip(self.lin, self.masks):
            m.copy_(((l.weight * m).abs() >= thr).float())
        n_grow = int(grow * keep)
        if n_grow <= 0:
            return
        gf = torch.cat([((l.weight.grad if l.weight.grad is not None
                          else torch.zeros_like(l.weight)).abs() * (1 - m)).flatten()
                        for l, m in zip(self.lin, self.masks)])
        if float(gf.max()) <= 0:
            return
        gt = torch.topk(gf, n_grow).values.min()
        off = 0
        for l, m in zip(self.lin, self.masks):
            sz = m.numel()
            add = (gf[off:off + sz].reshape(m.shape) >= gt) & (m == 0)
            m[add] = 1.0
            l.weight[add] = 0.0                            # born at zero, per RigL
            off += sz
        self.refresh_connectivity()   # dropout is conditioned on this; never stale


## 4. Training examples: every (context, perturbation), weighted by cells


In [ ]:
ctx_names = [c["name"] for c in ctxs]

# ---- repair degenerate baselines --------------------------------------------
# MEASURED: magpie_hipsc ships a baseline of ALL ZEROS (min 0, max 0, 0 nonzero
# genes of 18,080). Every other context has a real one. Since the context vector
# is log1p(baseline) / std, that context was handed 0/(0+1e-9) = the zero vector
# -- i.e. NO cell-state information -- and it is the training backbone, 44% of
# rows. Left alone the hypernetwork is conditioned on nothing for most of its
# data while H1 gets a real context, which makes the H1 comparison meaningless.
# Substitute the mean baseline of sibling contexts from the same study.
def _study_of(n):
    return "magpie" if "magpie" in n.lower() else n.split("::")[0]

_bad = [c for c in ctxs if float(np.abs(c["base"]).max()) == 0.0]
for c in _bad:
    sib = [d["base"] for d in ctxs
           if d is not c and _study_of(d["name"]) == _study_of(c["name"])
           and float(np.abs(d["base"]).max()) > 0]
    if sib:
        c["base"] = np.mean(np.stack(sib), 0).astype(np.float32)
        print(f"  REPAIRED baseline for {c['name']}: was all zeros, replaced with "
              f"the mean of {len(sib)} sibling contexts "
              f"({int((c['base'] != 0).sum()):,} nonzero genes)")
    else:
        print(f"  WARNING {c['name']} has an all-zero baseline and no sibling to "
              f"borrow from; its context vector carries no information")

ctx_vec = {}
for c in ctxs:
    b = np.log1p(np.maximum(c["base"], 0))
    sd = float(b.std())
    ctx_vec[c["name"]] = b / (sd + 1e-9) if sd > 1e-8 else np.zeros_like(b)
_deg = [n for n, v in ctx_vec.items() if float(np.abs(v).max()) == 0]
print(f"  context vectors: {len(ctx_vec) - len(_deg)}/{len(ctx_vec)} informative"
      + (f", DEGENERATE: {_deg}" if _deg else ""))
CTXD = 64
rpc = np.random.default_rng(1).normal(0, 1, (G, CTXD)).astype(np.float32) / math.sqrt(G)
CTXV = {k: torch.tensor(v @ rpc, device=dev) for k, v in ctx_vec.items()}
H1_ctx = torch.tensor((np.log1p(np.maximum(H1_base, 0)) /
                       (np.log1p(np.maximum(H1_base, 0)).std() + 1e-9)) @ rpc, device=dev)

rows = []
for ci, c in enumerate(ctxs):
    for i, p in enumerate(c["perts"]):
        if p in GPOS:
            rows.append((ci, i, GPOS[p]))
rows = np.array(rows)
print(f"{len(rows):,} (context, perturbation) training examples")

DELTA = [torch.tensor(c["delta"], device=dev) for c in ctxs]
NCELL = [torch.tensor(c["n"], dtype=torch.float32, device=dev) for c in ctxs]

def injection(gidx):
    """x0: knock the target gene down by its own baseline. The ONLY thing the
    perturbation does directly -- everything else is the simulation."""
    x0 = torch.zeros(len(gidx), G, device=dev)
    x0[torch.arange(len(gidx)), gidx] = -1.0
    return x0

# corpus activity prior: how much each gene actually moves, NON-H1 contexts only.
# ONE VOTE PER GENE, matching the training loss. The 19 magpie_line slices are
# replicate clones of one 444-gene panel; if rows are averaged directly they also
# overvote the gate/death prior even though their gradients are downweighted.
_gene_occ = np.bincount(rows[:, 2], minlength=G).astype(np.float64)
_act = np.zeros(G, np.float64)
for c in ctxs:
    _pg = np.array([GPOS.get(p, -1) for p in c["perts"]])
    _ok = _pg >= 0
    if not _ok.any():
        continue
    _ww = np.zeros(len(_pg), dtype=np.float64)
    _ww[_ok] = 1.0 / np.maximum(_gene_occ[_pg[_ok]], 1.0)
    _act += (np.abs(c["delta"].astype(np.float64)) * _ww[:, None]).sum(0)
_act = _act / max(int((_gene_occ > 0).sum()), 1)
ACT = torch.tensor(_act / (_act.mean() + 1e-12), dtype=torch.float32, device=dev)
# every gene that is ever a perturbation target, in the corpus or in H1
TARGET_GENES = ({int(g) for _, _, g in rows}
                | {GPOS[p] for p in H1_pert if p in GPOS})
print(f"  {len(TARGET_GENES):,} genes are perturbation targets -> immortal "
      f"(a target that dies makes its own perturbation unpredictable)")
print(f"corpus activity prior: median {float(ACT.median()):.4f}, "
      f"{float((ACT < 0.05).float().mean()) * 100:.1f}% of genes below 5% of mean")

def build_models(depth=None, hid=None, seed=SEED):
    """Fresh models. Factored out so width/depth can be SWEPT rather than guessed."""
    global plife, viab, hyper, opt
    torch.manual_seed(seed)
    d = DEPTH if depth is None else depth
    h = HID if hid is None else hid
    plife = Plife(D_GENE, M_MODES, T_STEPS).to(dev)
    viab = GeneViability(D_GENE, G, ACT, temp=GATE_TEMP, max_dead=MAX_DEAD,
                         protect=TARGET_GENES).to(dev)
    hyper = CrossLayerHyper(D_GENE + CTXD, h, d, M_MODES, DROPOUT,
                            spread=DROP_SPREAD).to(dev).to_dev(dev)
    opt = torch.optim.AdamW(list(plife.parameters()) + list(hyper.parameters())
                            + list(viab.parameters()), lr=LR, weight_decay=1e-4)
    n = sum(p.numel() for p in hyper.parameters())
    return n

# ---- replicate downweighting: one vote per GENE, spread over its contexts ----
from collections import Counter
gene_ctx = Counter()
for ci, pi, gi in rows:
    gene_ctx[gi] += 1
REPW = torch.tensor([1.0 / gene_ctx[g] for _, _, g in rows],
                    dtype=torch.float32, device=dev)
_mult = Counter(gene_ctx.values())
print(f"replicate structure: " + ", ".join(
    f"{n} genes seen {k}x" for k, n in sorted(_mult.items())[:4]) + " ...")
print(f"  core-panel genes carry {max(gene_ctx.values())}x the rows; downweighted "
      f"to one vote each")

# ---- holdout by DONOR, with the pooled backbone protected -------------------
# `magpie_pool` (= magpie_hipsc) is NOT a donor -- it is the pooled superset
# carrying 6,664 of the 7,585 unique genes. Holding it out leaves training with
# almost nothing but the 444-gene core panel and produces a held-out set LARGER
# than the training set. Measured on the first run:
#     holding out ['jejf','kolf','magpie_pool','oikd']
#       -> train 6,553 rows, held-out 8,594 rows   <- backwards
# It is the backbone; it always trains.
rng = np.random.default_rng(SEED)
# joung_tfatlas_hesc is also protected, for a different reason: H1 is an
# EMBRYONIC stem cell line and joung is the ONLY hESC context in the corpus
# (everything else is hiPSC). Holding it out removes the closest-matched
# context to the actual target from training, which is the opposite of what we
# want. The first run drew it as a "donor" and did exactly that.
PROTECTED = {"magpie_pool", "joung_tfatlas_hesc"}
donors = sorted({c["donor"] for c in ctxs})
cand = [d for d in donors if d not in PROTECTED]


HOLDOUT_MODE = "gene"      # "gene" (what VCC scores) or "donor" (diagnostic)
HOLDOUT_FRAC = 0.20


def set_holdout(seed=SEED, mode=None):
    """Split the corpus. DEFAULT IS BY GENE, and that is the whole point.

    MEASURED, and it changes which task we are even solving:

        cross-DONOR   ceiling  +0.1411 +- 0.1570   <- sd EXCEEDS the mean
        cross-GENE    oracle top-25 neighbours  0.2247
                      oracle top-100            0.2647

    The donor split asked "predict an unseen clone's response to a gene you
    HAVE seen" -- the log said it outright: `held-out genes NOT in train: 0`.
    Two donors measuring the same gene in the same lab agree at only 0.14, so
    that metric is mostly noise and cannot rank anything.

    VCC asks the other question: the 50 validation genes are NEVER perturbed in
    H1 training. That is cross-GENE, it is 3-4x more learnable, and it is what
    gets scored. Holding out genes makes the training signal match the target.
    """
    global ho, ho_donors, tr_rows, va_rows, _tr_map, _va_map, ho_genes
    mode = mode or HOLDOUT_MODE
    r = np.random.default_rng(seed)
    if mode == "gene":
        ug = np.array(sorted({int(g) for _, _, g in rows}))
        k = max(1, int(HOLDOUT_FRAC * len(ug)))
        ho_genes = set(r.choice(ug, k, replace=False).tolist())
        ho, ho_donors = set(), set()
        m = np.array([g in ho_genes for _, _, g in rows])
    else:
        ho_donors = {cand[i] for i in r.choice(len(cand),
                                               min(HOLDOUT_CTX, len(cand)),
                                               replace=False).tolist()}
        ho = {i for i, c in enumerate(ctxs) if c["donor"] in ho_donors}
        ho_genes = set()
        m = np.isin(rows[:, 0], list(ho))
    tr_rows, va_rows = rows[~m], rows[m]
    _a = np.arange(len(rows))
    _tr_map, _va_map = _a[~m], _a[m]
    return ho_genes if mode == "gene" else ho_donors


set_holdout(SEED)
print(f"HOLDOUT MODE: {HOLDOUT_MODE.upper()}  "
      f"(cross-gene ceiling ~0.26 vs cross-donor ~0.14)")
if HOLDOUT_MODE == "gene":
    print(f"  {len(ho_genes):,} genes held out ({100*HOLDOUT_FRAC:.0f}%), "
          f"every row for them removed from training")
else:
    print(f"  holding out donors {sorted(ho_donors)} -> {len(ho)} context files")
print(f"  train {len(tr_rows):,} rows / held-out {len(va_rows):,} rows")

_gtr = {g for _, _, g in tr_rows}
_gva = {g for _, _, g in va_rows}
print(f"  unique genes: train {len(_gtr):,}, held-out {len(_gva):,}, "
      f"held-out genes NOT in train: {len(_gva - _gtr):,}")
assert len(tr_rows) > len(va_rows), (
    f"held-out ({len(va_rows)}) exceeds train ({len(tr_rows)}) -- the split is "
    "backwards; check PROTECTED and HOLDOUT_CTX")

# HONEST CAVEAT, printed so it is not forgotten: magpie_hipsc is POOLED across
# donors, so a held-out donor's cells are still present inside it in aggregate.
# Donor holdout here therefore measures transfer to an unseen CLONE, not to a
# fully unseen genetic background. The clean cross-STUDY test is joung, which
# shares no cells with magpie at all.
if len(_gva - _gtr) == 0:
    print("  WARNING: every held-out gene also appears in training -- this is the "
          "cross-DONOR regime, ceiling 0.1411, mostly noise. Set "
          "HOLDOUT_MODE='gene'.")
else:
    print(f"  {len(_gva - _gtr):,} held-out genes are NEVER perturbed in training "
          f"-- this is the VCC task (cross-gene), ceiling ~0.26")


# PDS is a SAME-CONTEXT ranking task. VCC compares 50 perturbations measured in
# H1, so the training ranking batch must also compare perturbations within one
# context. Mixed-context batches measure context distance, not gene matching.
from collections import defaultdict


def build_context_pds_groups():
    global TR_BY_CTX, TR_CTX_IDS, TR_CTX_P
    TR_BY_CTX = defaultdict(list)
    for k, (ci, pi, gi) in enumerate(tr_rows):
        TR_BY_CTX[int(ci)].append(k)
    TR_CTX_IDS = np.array(sorted(TR_BY_CTX.keys()))
    # Rows x REPW gives every unique gene one vote. sqrt(rows) would still
    # over-represent genes from the 19 magpie_line clone files.
    TR_CTX_P = np.array([len(TR_BY_CTX[c]) for c in TR_CTX_IDS],
                        dtype=np.float64)
    TR_CTX_P = TR_CTX_P / TR_CTX_P.sum()


build_context_pds_groups()
print(f"  PDS ranking batches: {len(TR_CTX_IDS)} contexts, "
      f"sampled proportional to rows; with REPW this is one vote per gene")


## 5. Train — heavy early pruning, lightening; cross-layer regrowth


In [ ]:
def forward_rows(rr, idx_sel=None):
    """One forward pass shared by fitting, probes, and metrics."""
    if idx_sel is None:
        idx_sel = np.arange(len(rr))
    ci, pi, gi = rr[:, 0], rr[:, 1], rr[:, 2]
    gidx = torch.tensor(gi, device=dev)
    e = torch.cat([ZG[gidx], torch.stack([CTXV[ctx_names[c]] for c in ci])], 1)
    s, gamma, dt, gain, tau = hyper(e)
    tau = tau if USE_DYNAMIC_SPARSITY else None
    pred = plife(injection(gidx), s, gamma, dt, gain, ZG, viab.gate(ZG), tau=tau)
    y = torch.stack([DELTA[c][p] for c, p in zip(ci, pi)])
    w = torch.stack([NCELL[c][p] for c, p in zip(ci, pi)])
    w = w * REPW[idx_sel]
    w = w / w.mean()
    return pred, y, w, gidx


def pds_surrogate(pred, y, gidx, w, max_b=PDS_B):
    """Differentiable version of the exact PDS comparison.

    D[i,j] is the L1 distance from prediction i to truth j, with the ANCHOR's
    target column removed from every candidate. The default is a pairwise
    soft-margin loss; listwise CE remains available through PDS_LOSS="ce".
    """
    n = min(max_b, pred.shape[0])
    idx = torch.randperm(pred.shape[0], device=pred.device)[:n]
    p, yy, gg = pred[idx], y[idx], gidx[idx]

    D = torch.cdist(p, yy, p=1) / p.shape[1]
    pt = p.gather(1, gg[:, None]).squeeze(1)       # P_i[target_i]
    yt = yy[:, gg]                                  # Y_j[target_i]
    D = D - (pt[None, :] - yt).abs().T / p.shape[1]

    # Within-context duplicate genes are ambiguous, not useful negatives.
    same = gg[:, None].eq(gg[None, :])
    same.fill_diagonal_(False)
    eye = torch.eye(n, dtype=torch.bool, device=pred.device)
    valid = ~eye & ~same
    ww = w[idx]

    if PDS_LOSS == "ce":
        D_masked = D.masked_fill(same, 1e6)
        tau = (0.10 * D[valid].detach().median()).clamp_min(1e-6)
        per_anchor = F.cross_entropy(-D_masked / tau,
                                     torch.arange(n, device=pred.device),
                                     reduction="none")
    else:
        diag = D.diagonal().view(n, 1)
        scale = D[valid].detach().std().clamp_min(1e-6)
        margin = 0.10 * scale
        violation = (diag - D + margin) / scale
        pair_loss = F.softplus(violation)
        per_anchor = ((pair_loss * valid).sum(1) /
                      valid.sum(1).clamp_min(1))
    return (ww * per_anchor).sum() / ww.sum()


def context_pds_terms():
    """Deterministic PDS/shape pass among perturbations from ONE context."""
    was_h, was_p = hyper.training, plife.training
    hyper.eval(); plife.eval()       # ranking must not depend on dropout noise
    c = int(rng.choice(TR_CTX_IDS, p=TR_CTX_P))
    ids = np.array(TR_BY_CTX[c])
    take = rng.choice(len(ids), min(PDS_B, len(ids)), replace=False)
    ids = ids[take]
    rr = tr_rows[ids]
    pred, y, w, gidx = forward_rows(rr, _tr_map[ids])
    pds = pds_surrogate(pred, y, gidx, w, max_b=PDS_B)
    shape = (topk_shape_loss(pred, y, gidx, max_b=PDS_B)
             if USE_SHAPE_LOSS else torch.tensor(0.0, device=dev))
    if was_h: hyper.train()
    if was_p: plife.train()
    return pds, shape


def topk_shape_loss(pred, y, gidx, max_b=64,
                    ks=(10, 25, 50, 100, 250, 500, 1000)):
    """Match the cumulative L1 mass profile of real responses.

    PDS is an L1 ranking metric. A dense, mutually orthogonal profile can have a
    useful cosine signal and still be invisible to PDS. This term constrains the
    simulated response to put its mass into a realistic heavy tail.
    """
    n = min(max_b, pred.shape[0])
    idx = torch.randperm(pred.shape[0], device=pred.device)[:n]
    p = pred[idx].abs().clone()
    t = y[idx].abs().clone()
    gg = gidx[idx]
    p[torch.arange(n, device=dev), gg] = 0.0
    t[torch.arange(n, device=dev), gg] = 0.0
    kmax = min(max(ks), p.shape[1])
    ps = torch.topk(p, kmax, dim=1).values.cumsum(1)
    ts = torch.topk(t, kmax, dim=1).values.cumsum(1)
    kk = torch.tensor([k - 1 for k in ks if k <= kmax], device=dev)
    ps, ts = ps[:, kk], ts[:, kk]
    return ((ps - ts).abs() / ts.clamp_min(1e-6)).mean()


def batch_loss(rr, train=True, idx_sel=None, pds_w=0.0, shape_w=0.0):
    pred, y, w, gidx = forward_rows(rr, idx_sel)

    # THE TARGET GENE IS EXCLUDED FROM THE MAIN LOSS. PDS subtracts this column;
    # allowing it to dominate the fit produces the injection-spike failure mode.
    m = torch.ones_like(pred)
    m[torch.arange(len(gidx), device=dev), gidx] = 0.0
    po, yo = pred * m, y * m
    pn = po / (po.norm(dim=1, keepdim=True) + 1e-9)
    yn = yo / (yo.norm(dim=1, keepdim=True) + 1e-9)
    cos = 1 - (pn * yn).sum(1)
    l1 = (po - yo).abs().mean(1)
    tgt = (pred - y).gather(1, gidx[:, None]).abs().squeeze(1)
    loss = (w * (cos + 0.5 * l1 + 0.05 * tgt)).mean()

    if train and pds_w > 0 and USE_PDS_LOSS:
        loss = loss + pds_w * pds_surrogate(pred, y, gidx, w)
    if train and shape_w > 0 and USE_SHAPE_LOSS:
        loss = loss + shape_w * topk_shape_loss(pred, y, gidx)
    return loss


@torch.no_grad()
def hard_pds_batch(pred, y, gidx):
    """Exact PDS rank within this probe batch, using the scorer's target rule."""
    n = pred.shape[0]
    D = torch.cdist(pred, y, p=1)
    pt = pred.gather(1, gidx[:, None]).squeeze(1)
    yt = y[:, gidx]
    D = D - (pt[None, :] - yt).abs().T
    r = torch.argsort(torch.argsort(D, dim=1), dim=1)
    return float((1.0 - r[torch.arange(n, device=dev), torch.arange(n, device=dev)].float()
                  / max(n - 1, 1)).mean())


@torch.no_grad()
def lifecycle_probe(rr, idx_sel):
    """Fixed-batch metrics used before/after every lifecycle edit."""
    was_h, was_p = hyper.training, plife.training
    hyper.eval(); plife.eval()
    pred, y, w, gidx = forward_rows(rr, idx_sel)
    msk = torch.ones_like(pred)
    msk[torch.arange(len(gidx), device=dev), gidx] = 0.0
    po, yo = pred * msk, y * msk
    a = po - po.mean(1, keepdim=True)
    b = yo - yo.mean(1, keepdim=True)
    corr = ((a * b).sum(1) /
            (a.norm(dim=1) * b.norm(dim=1) + 1e-9)).mean()
    out = {
        "pds": hard_pds_batch(pred, y, gidx),
        "mae": float((pred - y).abs().mean()),
        "corr": float(corr),
        "l1": float(pred.abs().mean()),
        "zero": float((pred == 0).float().mean()),
    }
    if was_h: hyper.train()
    if was_p: plife.train()
    return out


@torch.no_grad()
def accuracy(rr, idx_sel):
    """Mean per-perturbation off-target correlation."""
    hyper.eval(); plife.eval()
    pred, y, w, gidx = forward_rows(rr, idx_sel)
    msk = torch.ones_like(pred)
    msk[torch.arange(len(gidx), device=dev), gidx] = 0.0
    pred, y = pred * msk, y * msk
    a = pred - pred.mean(1, keepdim=True)
    b = y - y.mean(1, keepdim=True)
    return float(((a * b).sum(1) /
                  (a.norm(dim=1) * b.norm(dim=1) + 1e-9)).mean())


HIST = {k: [] for k in ("ep", "train", "val", "acc_tr", "acc_va", "density",
                        "units_dead", "genes_dead", "skips", "pd_lo", "pd_hi",
                        "coupling", "pds_w", "pds_ce", "shape_loss", "probe_pds",
                        "drive_zero", "life_dpds", "life_dmae", "life_dcorr")}
hist = []
LAST_LIFE_DELTA = {"pds": 0.0, "mae": 0.0, "corr": 0.0}


def train_models(epochs, record=True, verbose=True, every=25):
    """One training run. Returns (final held-out loss, final held-out accuracy)."""
    global n_skip, skip_total, LAST_LIFE_DELTA, LAST_PDS_CE, LAST_SHAPE
    build_context_pds_groups()       # refresh after any rotated holdout
    n_skip = 0
    skip_total = 0
    LAST_PDS_CE = 0.0
    LAST_SHAPE = 0.0
    t0 = time.time()
    vl = float("nan")
    acc_va = float("nan")

    # Fixed probe rows make lifecycle deltas comparable across events.
    _lr = np.random.default_rng(SEED + 999)
    _li = _lr.choice(len(tr_rows), min(LIFECYCLE_B, len(tr_rows)), replace=False)
    life_rows, life_map = tr_rows[_li], _tr_map[_li]

    for ep in range(epochs):
        hyper.train(); plife.train()
        t = ep / max(epochs - 1, 1)
        ramp = min(1.0, max(0.0, (t - PDS_WARMUP) / max(PDS_RAMP, 1e-9)))
        pds_w = PDS_W * ramp if USE_PDS_LOSS else 0.0
        shape_w = SHAPE_W * ramp if USE_SHAPE_LOSS else 0.0

        _si = rng.choice(len(tr_rows), min(BATCH, len(tr_rows)), replace=False)
        sel = tr_rows[_si]
        # Pointwise fit stays mixed-context; ranking/shape are same-context.
        loss = batch_loss(sel, idx_sel=_tr_map[_si], pds_w=0.0,
                          shape_w=0.0)
        if pds_w > 0 or shape_w > 0:
            pds_term, shape_term = context_pds_terms()
            if pds_w > 0:
                loss = loss + pds_w * pds_term
                LAST_PDS_CE = float(pds_term.detach())
            if shape_w > 0:
                loss = loss + shape_w * shape_term
                LAST_SHAPE = float(shape_term.detach())
        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"non-finite loss at epoch {ep} (depth={hyper.depth}, "
                f"width={hyper.hid}). Aborting instead of running to completion "
                f"and reporting nan. Check: field scale, gene gate, gain_f, tau.")
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(plife.parameters()) + list(hyper.parameters())
            + list(viab.parameters()), 1.0)
        _bad = [n for n, q in list(plife.named_parameters())
                + list(hyper.named_parameters())
                if q.grad is not None and not torch.isfinite(q.grad).all()]
        if _bad:
            raise FloatingPointError(f"non-finite GRADIENT at epoch {ep} in "
                                     f"{_bad[:4]} -- loss was finite, so this is "
                                     f"a backward-pass singularity, not a blowup")

        # connectivity means nothing before pruning starts, so phase the coupling in
        hyper.coupling = min(1.0, max(0.0, (t - 0.10) / 0.25))
        if ep % 10 == 0 and t < 0.85:
            dens = DENSITY + (1 - DENSITY) * max(0.0, 1 - 3 * t)
            grow = 0.3 * 0.5 * (1 + math.cos(math.pi * min(1.0, t / 0.75)))
            do_life = (USE_PRUNE or (USE_STREAMLINE and t >= STREAMLINE_WARMUP)
                       or USE_UNIT_DEATH or USE_GENE_DEATH)
            pre = lifecycle_probe(life_rows, life_map) if (LIFECYCLE_AUDIT and do_life) else None

            if USE_PRUNE:
                hyper.prune_and_grow(dens, grow if USE_REGROW else 0.0)
            if USE_STREAMLINE and t >= STREAMLINE_WARMUP:
                n_skip = hyper.streamline()
                skip_total += n_skip
            else:
                n_skip = 0
            if USE_UNIT_DEATH:
                hyper.reap_units(t)
            if USE_GENE_DEATH:
                viab.reap(ZG, t)

            if pre is not None:
                post = lifecycle_probe(life_rows, life_map)
                LAST_LIFE_DELTA = {k: post[k] - pre[k] for k in ("pds", "mae", "corr")}
            else:
                LAST_LIFE_DELTA = {"pds": 0.0, "mae": 0.0, "corr": 0.0}
        else:
            n_skip = 0
            LAST_LIFE_DELTA = {"pds": 0.0, "mae": 0.0, "corr": 0.0}
        opt.step()

        if ep % every == 0 or ep == epochs - 1:
            hyper.eval(); plife.eval()
            with torch.no_grad():
                _vi = rng.choice(len(va_rows), min(1024, len(va_rows)), replace=False)
                vl = float(batch_loss(va_rows[_vi], False, idx_sel=_va_map[_vi]))
            _si2 = rng.choice(len(tr_rows), min(512, len(tr_rows)), replace=False)
            acc_va = accuracy(va_rows[_vi], _va_map[_vi])
            _probe_now = lifecycle_probe(life_rows, life_map)
            if record:
                pd = hyper._pdrop[hyper.depth]
                HIST["ep"].append(ep); HIST["train"].append(float(loss.detach()))
                HIST["val"].append(vl)
                HIST["acc_tr"].append(accuracy(tr_rows[_si2], _tr_map[_si2]))
                HIST["acc_va"].append(acc_va)
                HIST["density"].append(hyper.density())
                HIST["units_dead"].append(hyper.n_units_dead())
                HIST["genes_dead"].append(viab.n_dead())
                HIST["skips"].append(skip_total)
                HIST["pd_lo"].append(float(pd.min()))
                HIST["pd_hi"].append(float(pd.max()))
                HIST["coupling"].append(hyper.coupling)
                HIST["pds_w"].append(pds_w)
                HIST["pds_ce"].append(LAST_PDS_CE)
                HIST["shape_loss"].append(LAST_SHAPE)
                HIST["probe_pds"].append(_probe_now["pds"])
                HIST["drive_zero"].append(plife._drive_zero)
                HIST["life_dpds"].append(LAST_LIFE_DELTA["pds"])
                HIST["life_dmae"].append(LAST_LIFE_DELTA["mae"])
                HIST["life_dcorr"].append(LAST_LIFE_DELTA["corr"])
                hist.append((ep, float(loss.detach()), vl, hyper.density()))
            if verbose:
                print(f"  ep {ep:>4}  train {float(loss.detach()):.4f}  "
                      f"held-out-{HOLDOUT_MODE} {vl:.4f}  acc {acc_va:+.4f}  "
                      f"pds_w {pds_w:.3f}  pdsL {LAST_PDS_CE:.3f}  "
                      f"probePDS {_probe_now['pds']:.4f}  "
                      f"drive0 {100*plife._drive_zero:4.1f}%  "
                      f"density {hyper.density():.3f}  "
                      f"units {hyper.n_units_dead():>3} dead  "
                      f"(zero {hyper.tiers[0]} under {hyper.tiers[1]})  "
                      f"skips +{n_skip}  genes {viab.n_dead():>5} dead  "
                      f"life dPDS {LAST_LIFE_DELTA['pds']:+.4f} "
                      f"dMAE {LAST_LIFE_DELTA['mae']:+.5f}  "
                      f"[{time.time()-t0:.0f}s]")
    return vl, acc_va


## Contrastive matcher probe and retrieval bank

The plife margin run stayed at chance, so this version separates **matching**
from **generation**. A small contrastive network asks whether the fixed gene
descriptor plus context can identify the true target-masked response profile.
It trains only on corpus training genes, uses corpus held-out genes for early
stopping, and never reads validation truth.

If matching transfers to H1, the generator is no longer the bottleneck: the
same matcher can retrieve and blend real corpus response profiles. The final
choice remains constrained on H1 training: MAE and DES cannot regress relative
to the safe median-plus-DE-prior incumbent.


In [ ]:

# ============================================================
# CONTRASTIVE MATCHER PROBE -- representation diagnostic
#
# This does NOT train or modify hyper/plife/viab. It asks whether
# (ZG[target gene], context) can identify the TRUE same-context response.
# Corpus held-out genes are used for early stopping. Validation stays closed.
# ============================================================
from collections import defaultdict

MATCH_B = 64
MATCH_MIN_B = 16
MATCH_STEPS = 1500          # 300 for smoke; 1500-3000 for the real probe
MATCH_LR = 1e-3
MATCH_WD = 1e-4
MATCH_WIDTH = 256
MATCH_DIM = 128
MATCH_TEMP = 0.07
MATCH_EVAL_BATCHES = 100
MATCH_LOG_EVERY = 250

assert HOLDOUT_MODE == "gene", "Run this probe with HOLDOUT_MODE='gene'."
if SCORE_VALIDATION:
    print("  validation truth remains closed until the final scoring cell; matcher does not use it")


class MatcherProbe(nn.Module):
    def __init__(self, d_gene, d_ctx, d_response, width=256, d=128):
        super().__init__()
        self.query = nn.Sequential(
            nn.Linear(d_gene + d_ctx, width),
            nn.GELU(),
            nn.Linear(width, width),
            nn.GELU(),
            nn.Linear(width, d),
        )
        self.key = nn.Sequential(
            nn.Linear(d_response, width),
            nn.GELU(),
            nn.Linear(width, width),
            nn.GELU(),
            nn.Linear(width, d),
        )

    def forward(self, z_gene, ctx, y_off):
        q = self.query(torch.cat([z_gene, ctx], dim=1))
        k = self.key(y_off)
        return F.normalize(q, dim=1), F.normalize(k, dim=1)


def _matcher_ctx_index(split_rows, by_ctx):
    ids, ps = [], []
    for c in sorted(by_ctx):
        unique_genes = {int(split_rows[k, 2]) for k in by_ctx[c]}
        if len(unique_genes) >= MATCH_MIN_B:
            ids.append(int(c))
            ps.append(len(by_ctx[c]))
    ids = np.asarray(ids, dtype=int)
    ps = np.asarray(ps, dtype=np.float64)
    return ids, ps / ps.sum()


def _take_unique_rows(split_rows, ids, n, r):
    order = r.permutation(len(ids))
    seen, take = set(), []
    for j in order:
        row_id = int(ids[j])
        g = int(split_rows[row_id, 2])
        if g in seen:
            continue
        seen.add(g)
        take.append(row_id)
        if len(take) == n:
            break
    return np.asarray(take, dtype=int)


def matcher_batch(split_rows, split_map, ctx_ids, ctx_p, by_ctx, r):
    for _ in range(50):
        c = int(r.choice(ctx_ids, p=ctx_p))
        take = _take_unique_rows(split_rows, np.asarray(by_ctx[c], dtype=int),
                                 MATCH_B, r)
        if len(take) >= MATCH_MIN_B:
            break
    else:
        raise RuntimeError("No context has enough unique genes for matcher.")

    rr = split_rows[take]
    maps = split_map[take]
    ci = rr[:, 0].astype(int)
    pi = rr[:, 1].astype(int)
    gi = rr[:, 2].astype(int)

    gidx = torch.as_tensor(gi, dtype=torch.long, device=dev)
    z_gene = ZG[gidx]
    ctx = torch.stack([CTXV[ctx_names[int(x)]] for x in ci])
    y_off = torch.stack([DELTA[int(c)][int(p)] for c, p in zip(ci, pi)]).clone()
    y_off[torch.arange(len(gidx), device=dev), gidx] = 0.0

    map_t = torch.as_tensor(maps, dtype=torch.long, device=dev)
    w = REPW[map_t].detach()
    return z_gene, ctx, y_off, w, gidx


def matcher_loss(q, k, w):
    logits = (q @ k.T) / MATCH_TEMP
    labels = torch.arange(q.shape[0], device=dev)
    loss_q = F.cross_entropy(logits, labels, reduction="none")
    loss_k = F.cross_entropy(logits.T, labels, reduction="none")
    return (w * (loss_q + loss_k)).sum() / (2.0 * w.sum().clamp_min(1e-9))


@torch.no_grad()
def _embedding_rank(q, k):
    n = q.shape[0]
    D = torch.cdist(q, k, p=1)
    r = torch.argsort(torch.argsort(D, dim=1), dim=1)
    diag = r[torch.arange(n, device=dev), torch.arange(n, device=dev)].float()
    pds = float((1.0 - diag / max(n - 1, 1)).mean())
    top1 = float((D.argmin(dim=1) == torch.arange(n, device=dev)).float().mean())
    return pds, top1, n


@torch.no_grad()
def matcher_eval(split_rows, split_map, ctx_ids, ctx_p, by_ctx,
                 batches=MATCH_EVAL_BATCHES, seed=SEED + 991):
    was_training = matcher.training
    matcher.eval()
    r = np.random.default_rng(seed)
    vals, tops, ns = [], [], []
    for _ in range(batches):
        try:
            z_gene, ctx, y_off, _, _ = matcher_batch(
                split_rows, split_map, ctx_ids, ctx_p, by_ctx, r)
        except RuntimeError:
            break
        q, k = matcher(z_gene, ctx, y_off)
        pds, top1, n = _embedding_rank(q, k)
        vals.append(pds); tops.append(top1); ns.append(n)
    if was_training:
        matcher.train()
    if not vals:
        return float("nan"), float("nan"), 0
    return float(np.mean(vals)), float(np.mean(tops)), int(np.mean(ns))


VA_BY_CTX = defaultdict(list)
for k, (ci, pi, gi) in enumerate(va_rows):
    VA_BY_CTX[int(ci)].append(k)

TR_M_CTX, TR_M_P = _matcher_ctx_index(tr_rows, TR_BY_CTX)
VA_M_CTX, VA_M_P = _matcher_ctx_index(va_rows, VA_BY_CTX)
print(f"  matcher contexts: train {len(TR_M_CTX)}, held-out {len(VA_M_CTX)}")

matcher = MatcherProbe(D_GENE, CTXD, G, MATCH_WIDTH, MATCH_DIM).to(dev)
mopt = torch.optim.AdamW(matcher.parameters(), lr=MATCH_LR, weight_decay=MATCH_WD)
match_rng = np.random.default_rng(SEED + 711)
best_va_pds = -1.0
best_state = None

for step in range(1, MATCH_STEPS + 1):
    matcher.train()
    z_gene, ctx, y_off, w, _ = matcher_batch(
        tr_rows, _tr_map, TR_M_CTX, TR_M_P, TR_BY_CTX, match_rng)
    q, k = matcher(z_gene, ctx, y_off)
    loss = matcher_loss(q, k, w)
    if not torch.isfinite(loss):
        raise FloatingPointError(f"matcher loss became non-finite at step {step}")

    mopt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(matcher.parameters(), 1.0)
    mopt.step()

    if step % MATCH_LOG_EVERY == 0 or step == MATCH_STEPS:
        tr_p, tr_t, tr_n = matcher_eval(
            tr_rows, _tr_map, TR_M_CTX, TR_M_P, TR_BY_CTX,
            batches=10, seed=SEED + 1000 + step)
        va_p, va_t, va_n = matcher_eval(
            va_rows, _va_map, VA_M_CTX, VA_M_P, VA_BY_CTX,
            batches=10, seed=SEED + 2000 + step)
        if np.isfinite(va_p) and va_p > best_va_pds:
            best_va_pds = va_p
            best_state = {k: v.detach().cpu().clone()
                          for k, v in matcher.state_dict().items()}
        print(f"  matcher step {step:>4}  loss {float(loss.detach()):.4f}  "
              f"trainPDS {tr_p:.4f} top1 {tr_t:.3f}  "
              f"heldGenePDS {va_p:.4f} top1 {va_t:.3f}  "
              f"best {best_va_pds:.4f}")

# Early stop on the legal corpus held-out split. This guards against the large
# train/held-out gap observed in the first matcher run.
if best_state is not None:
    matcher.load_state_dict(best_state)
    matcher.to(dev)
    print(f"  restored matcher state with held-out corpus PDS {best_va_pds:.4f}")

train_match = matcher_eval(tr_rows, _tr_map, TR_M_CTX, TR_M_P, TR_BY_CTX,
                           batches=MATCH_EVAL_BATCHES, seed=SEED + 3001)
held_match = matcher_eval(va_rows, _va_map, VA_M_CTX, VA_M_P, VA_BY_CTX,
                          batches=MATCH_EVAL_BATCHES, seed=SEED + 3002)

_H1_G = np.array([GPOS.get(p, -1) for p in H1_pert], dtype=int)
_H1_OK = np.where(_H1_G >= 0)[0]
_H1_Y = torch.as_tensor(H1_lfc - H1_lfc.mean(0, keepdims=True),
                        dtype=torch.float32, device=dev)


def _take_unique_h1(sel, n, r):
    order = r.permutation(len(sel))
    seen, take = set(), []
    for j in order:
        row_id = int(sel[j])
        g = int(_H1_G[row_id])
        if g in seen:
            continue
        seen.add(g)
        take.append(row_id)
        if len(take) == n:
            break
    return np.asarray(take, dtype=int)


@torch.no_grad()
def matcher_h1_eval(sel, batches=MATCH_EVAL_BATCHES, seed=SEED + 4000):
    sel = np.asarray(sel, dtype=int)
    if len(sel) == 0 or len(np.unique(_H1_G[sel])) < MATCH_MIN_B:
        return float("nan"), float("nan"), 0
    was_training = matcher.training
    matcher.eval()
    r = np.random.default_rng(seed)
    vals, tops, ns = [], [], []
    for _ in range(batches):
        take = _take_unique_h1(sel, min(MATCH_B, len(sel)), r)
        if len(take) < MATCH_MIN_B:
            continue
        gidx = torch.as_tensor(_H1_G[take], dtype=torch.long, device=dev)
        z_gene = ZG[gidx]
        ctx = H1_ctx[None, :].repeat(len(gidx), 1)
        y_off = _H1_Y[take].clone()
        y_off[torch.arange(len(gidx), device=dev), gidx] = 0.0
        q, k = matcher(z_gene, ctx, y_off)
        pds, top1, n = _embedding_rank(q, k)
        vals.append(pds); tops.append(top1); ns.append(n)
    if was_training:
        matcher.train()
    if not vals:
        return float("nan"), float("nan"), 0
    return float(np.mean(vals)), float(np.mean(tops)), int(np.mean(ns))


h1_all = matcher_h1_eval(_H1_OK, seed=SEED + 4001)
_h1_unseen = np.array([i for i in _H1_OK if int(_H1_G[i]) in ho_genes], dtype=int)
h1_unseen = matcher_h1_eval(_h1_unseen, seed=SEED + 4002)

print("\nMATCHER PROBE RESULT")
print(f"  corpus train genes:       PDS {train_match[0]:.4f}  "
      f"top1 {train_match[1]:.3f}  n~{train_match[2]}")
print(f"  corpus held-out genes:    PDS {held_match[0]:.4f}  "
      f"top1 {held_match[1]:.3f}  n~{held_match[2]}")
print(f"  H1 train, all genes:      PDS {h1_all[0]:.4f}  "
      f"top1 {h1_all[1]:.3f}  n~{h1_all[2]}")
print(f"  H1 train, unseen genes:   PDS {h1_unseen[0]:.4f}  "
      f"top1 {h1_unseen[1]:.3f}  n~{h1_unseen[2]}")
print("  chance PDS ~= 0.5000")


In [ ]:

# ============================================================
# MATCHER RETRIEVAL PREDICTOR
#
# First result: matcher embedding PDS transferred, but K=8 cosine retrieval
# blended the signal away. Two fixes are tested LEGALLY on corpus holdout:
#   1. retrieve with the same L1 embedding geometry used by the matcher probe;
#   2. sharpen K/temperature instead of averaging many profiles into a median.
# ============================================================

MATCH_RETRIEVE_MODE = "l1"       # "l1" matches the matcher PDS geometry
MATCH_RETRIEVE_K = 1
MATCH_RETRIEVE_TEMP = 0.50       # used only when K > 1
MATCH_RETRIEVE_EXCLUDE_SELF = True
MATCH_BANK_BATCH = 128
RETRIEVAL_PROBE_BATCHES = 50
RETRIEVAL_SWEEP_BATCHES = 30


@torch.no_grad()
def build_matcher_bank():
    """Embed every corpus TRAINING response; held-out corpus genes stay out."""
    matcher.eval()
    keys, ys, genes, maps = [], [], [], []
    for start in range(0, len(tr_rows), MATCH_BANK_BATCH):
        rr = tr_rows[start:start + MATCH_BANK_BATCH]
        mp = _tr_map[start:start + MATCH_BANK_BATCH]
        ci = rr[:, 0].astype(int)
        pi = rr[:, 1].astype(int)
        gi = rr[:, 2].astype(int)
        gidx = torch.as_tensor(gi, dtype=torch.long, device=dev)
        z_gene = ZG[gidx]
        ctx = torch.stack([CTXV[ctx_names[int(x)]] for x in ci])
        y_off = torch.stack([DELTA[int(c)][int(p)]
                             for c, p in zip(ci, pi)]).clone()
        y_off[torch.arange(len(gidx), device=dev), gidx] = 0.0
        _, k = matcher(z_gene, ctx, y_off)
        keys.append(k.cpu())
        ys.append(y_off.cpu().numpy().astype(np.float16))
        genes.append(gi)
        maps.append(mp)
    return (torch.cat(keys).to(dev),
            np.concatenate(ys),
            np.concatenate(genes).astype(int),
            np.concatenate(maps).astype(int))


MATCH_BANK_K, MATCH_BANK_Y, MATCH_BANK_G, MATCH_BANK_MAP = build_matcher_bank()
print(f"  retrieval bank: {len(MATCH_BANK_G):,} corpus training profiles, "
      f"{len(set(MATCH_BANK_G.tolist())):,} unique source genes")


@torch.no_grad()
def _matcher_queries_for_genes(gene_ids, ctxvec, batch=256):
    gene_ids = np.asarray(gene_ids, dtype=int)
    qs = []
    matcher.eval()
    for start in range(0, len(gene_ids), batch):
        gg = gene_ids[start:start + batch]
        safe = np.maximum(gg, 0)
        gidx = torch.as_tensor(safe, dtype=torch.long, device=dev)
        z_gene = ZG[gidx]
        ctx = ctxvec[None, :].repeat(len(gidx), 1)
        q = F.normalize(matcher.query(torch.cat([z_gene, ctx], dim=1)), dim=1)
        qs.append(q.cpu())
    return torch.cat(qs).to(dev)


@torch.no_grad()
def _retrieve_from_queries(q, target_genes, K=MATCH_RETRIEVE_K,
                           temp=MATCH_RETRIEVE_TEMP,
                           mode=MATCH_RETRIEVE_MODE,
                           exclude_self=MATCH_RETRIEVE_EXCLUDE_SELF):
    """Top-K unique source genes. mode='l1' uses the same geometry as the probe."""
    if mode == "l1":
        scores = (-torch.cdist(q, MATCH_BANK_K, p=1)).detach().cpu().numpy()
    elif mode == "dot":
        scores = (q @ MATCH_BANK_K.T).detach().cpu().numpy()
    else:
        raise ValueError(f"unknown retrieval mode {mode!r}")

    target_genes = np.asarray(target_genes, dtype=int)
    out = np.zeros((len(q), G), dtype=np.float32)

    for i in range(len(q)):
        if target_genes[i] < 0:
            continue
        s = scores[i].copy()
        if exclude_self:
            s[MATCH_BANK_G == int(target_genes[i])] = -np.inf

        order = np.argsort(-s)
        seen, take = set(), []
        for j in order:
            if not np.isfinite(s[j]):
                continue
            source_gene = int(MATCH_BANK_G[j])
            if source_gene in seen:
                continue
            seen.add(source_gene)
            take.append(int(j))
            if len(take) == K:
                break
        if not take:
            continue

        ss = s[take]
        ss = (ss - ss.max()) / max(float(temp), 1e-6)
        w = np.exp(ss)
        w = w / w.sum()
        Y = MATCH_BANK_Y[take].astype(np.float32)
        out[i] = w @ Y
        out[i, int(target_genes[i])] = 0.0
    return out


def matcher_retrieve(pert_names, ctxvec, K=MATCH_RETRIEVE_K,
                     temp=MATCH_RETRIEVE_TEMP, mode=MATCH_RETRIEVE_MODE,
                     exclude_self=MATCH_RETRIEVE_EXCLUDE_SELF):
    gene_ids = np.array([GPOS.get(p, -1) for p in pert_names], dtype=int)
    q = _matcher_queries_for_genes(gene_ids, ctxvec)
    return _retrieve_from_queries(q, gene_ids, K=K, temp=temp, mode=mode,
                                  exclude_self=exclude_self)


@torch.no_grad()
def retrieval_corpus_probe(K=MATCH_RETRIEVE_K, temp=MATCH_RETRIEVE_TEMP,
                           mode=MATCH_RETRIEVE_MODE,
                           batches=RETRIEVAL_PROBE_BATCHES,
                           seed=SEED + 5100):
    """Original-G-space PDS of retrieved profiles on held-out corpus genes."""
    matcher.eval()
    r = np.random.default_rng(seed)
    vals, maes, corrs = [], [], []
    for _ in range(batches):
        z_gene, ctx, y_off, _, gidx = matcher_batch(
            va_rows, _va_map, VA_M_CTX, VA_M_P, VA_BY_CTX, r)
        x = torch.cat([z_gene, ctx], dim=1)
        q = F.normalize(matcher.query(x), dim=1)
        pred = _retrieve_from_queries(q, gidx.cpu().numpy(),
                                      K=K, temp=temp, mode=mode)
        pred = torch.as_tensor(pred, dtype=torch.float32, device=dev)

        vals.append(hard_pds_batch(pred, y_off, gidx))
        maes.append(float((pred - y_off).abs().mean()))
        a = pred - pred.mean(1, keepdim=True)
        b = y_off - y_off.mean(1, keepdim=True)
        corrs.append(float(((a * b).sum(1) /
                            (a.norm(dim=1) * b.norm(dim=1) + 1e-9)).mean()))
    return {"pds": float(np.mean(vals)), "mae": float(np.mean(maes)),
            "corr": float(np.mean(corrs)), "K": K, "temp": temp, "mode": mode}


# Same fixed probe batches for every candidate, selected without H1 or validation.
_sweep_specs = [
    ("l1", 1, 0.50),
    ("l1", 2, 0.50),
    ("l1", 4, 0.50),
    ("l1", 4, 1.00),
    ("l1", 8, 0.50),
    ("dot", 1, 0.10),
    ("dot", 4, 0.10),
    ("dot", 8, 0.10),       # previous failed setting
]
_sweep = []
print("\n  retrieval geometry sweep on held-out corpus genes")
print(f"  {'mode':>5}{'K':>4}{'temp':>7}{'PDS':>9}{'MAE':>10}{'corr':>9}")
for _mode, _k, _temp in _sweep_specs:
    _r = retrieval_corpus_probe(K=_k, temp=_temp, mode=_mode,
                                batches=RETRIEVAL_SWEEP_BATCHES)
    _sweep.append(_r)
    print(f"  {_mode:>5}{_k:>4}{_temp:>7.2f}{_r['pds']:>9.4f}"
          f"{_r['mae']:>10.5f}{_r['corr']:>9.4f}")

_best = max(_sweep, key=lambda x: (x["pds"], x["corr"], -x["mae"]))
MATCH_RETRIEVE_MODE = _best["mode"]
MATCH_RETRIEVE_K = int(_best["K"])
MATCH_RETRIEVE_TEMP = float(_best["temp"])
print(f"\n  selected retrieval: mode={MATCH_RETRIEVE_MODE}, "
      f"K={MATCH_RETRIEVE_K}, temp={MATCH_RETRIEVE_TEMP:.2f}")

retr_probe = retrieval_corpus_probe(K=MATCH_RETRIEVE_K,
                                    temp=MATCH_RETRIEVE_TEMP,
                                    mode=MATCH_RETRIEVE_MODE,
                                    batches=RETRIEVAL_PROBE_BATCHES,
                                    seed=SEED + 9100)
print(f"  retrieval, held-out corpus genes: PDS {retr_probe['pds']:.4f}  "
      f"off-target MAE {retr_probe['mae']:.5f}  corr {retr_probe['corr']:+.4f}")

# Legal H1-training predictions used for constrained selection below.
H1_match = matcher_retrieve(H1_pert, H1_ctx)
np.savez_compressed("/kaggle/working/matcher_h1_train.npz",
                    R=H1_match.astype(np.float32), perts=np.array(H1_pert))
print(f"  wrote /kaggle/working/matcher_h1_train.npz {H1_match.shape}  "
      f"(retrieval mode={MATCH_RETRIEVE_MODE}, K={MATCH_RETRIEVE_K}, "
      f"temp={MATCH_RETRIEVE_TEMP})")


## Matcher-conditioned response decoder

Direct retrieval did not preserve the matcher signal: every retrieval geometry
stayed near PDS 0.50. The next test therefore stops copying corpus profiles and
trains a deep decoder from the frozen matcher query embedding to the full
18,080-gene response. It uses pointwise reconstruction, same-context PDS, and
shape losses, with early stopping on held-out corpus genes. Validation remains
closed.


In [ ]:

# ============================================================
# MATCHER-CONDITIONED RESPONSE DECODER
#
# Retrieval failed because nearest keys are not necessarily close responses in
# original L1 space. This decoder instead learns q(target, context) -> full
# response profile. The matcher is frozen; only the decoder trains.
# ============================================================

DECODER_STEPS = 1200          # 300 for smoke; 1200-2500 for real run
DECODER_BATCH = 128
DECODER_LR = 1e-3
DECODER_WD = 1e-4
DECODER_WIDTH = 512
DECODER_PDS_W = 0.25
DECODER_SHAPE_W = 0.05
DECODER_RAMP = 0.10
DECODER_LOG_EVERY = 100
DECODER_PROBE_BATCHES = 20
DECODER_FINAL_PROBES = 50

for _p in matcher.parameters():
    _p.requires_grad_(False)
matcher.eval()


class ResponseDecoder(nn.Module):
    def __init__(self, d=128, width=512, n_genes=G):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, width),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(width, width),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(width, n_genes),
        )
        last = self.net[-1]
        nn.init.zeros_(last.weight)
        nn.init.zeros_(last.bias)

    def forward(self, q):
        return self.net(q)


decoder = ResponseDecoder(MATCH_DIM, DECODER_WIDTH, G).to(dev)
dopt = torch.optim.AdamW(decoder.parameters(), lr=DECODER_LR,
                         weight_decay=DECODER_WD)
decoder_rng = np.random.default_rng(SEED + 6200)


@torch.no_grad()
def frozen_matcher_query(z_gene, ctx):
    matcher.eval()
    q = matcher.query(torch.cat([z_gene, ctx], dim=1))
    return F.normalize(q, dim=1)


def decoder_row_batch(split_rows, split_map, n, r):
    ids = r.choice(len(split_rows), min(n, len(split_rows)), replace=False)
    rr = split_rows[ids]
    maps = split_map[ids]
    ci = rr[:, 0].astype(int)
    pi = rr[:, 1].astype(int)
    gi = rr[:, 2].astype(int)
    gidx = torch.as_tensor(gi, dtype=torch.long, device=dev)
    z_gene = ZG[gidx]
    ctx = torch.stack([CTXV[ctx_names[int(c)]] for c in ci])
    y = torch.stack([DELTA[int(c)][int(p)] for c, p in zip(ci, pi)]).clone()
    y[torch.arange(len(gidx), device=dev), gidx] = 0.0
    q = frozen_matcher_query(z_gene, ctx)
    w = torch.stack([NCELL[int(c)][int(p)] for c, p in zip(ci, pi)])
    map_t = torch.as_tensor(maps, dtype=torch.long, device=dev)
    w = (w * REPW[map_t]).detach()
    w = w / w.mean().clamp_min(1e-9)
    return q, y, w, gidx


def decoder_pointwise_loss(q, y, w, gidx):
    pred = decoder(q)
    pred = pred.clone()
    pred[torch.arange(len(gidx), device=dev), gidx] = 0.0
    pn = pred / (pred.norm(dim=1, keepdim=True) + 1e-9)
    yn = y / (y.norm(dim=1, keepdim=True) + 1e-9)
    cos = 1.0 - (pn * yn).sum(1)
    l1 = (pred - y).abs().mean(1)
    return (w * (cos + 0.5 * l1)).mean()


def decoder_context_terms():
    was_training = decoder.training
    decoder.eval()             # deterministic ranking pass, gradients preserved
    z_gene, ctx, y, w, gidx = matcher_batch(
        tr_rows, _tr_map, TR_M_CTX, TR_M_P, TR_BY_CTX, decoder_rng)
    q = frozen_matcher_query(z_gene, ctx)
    pred = decoder(q)
    pred = pred.clone()
    pred[torch.arange(len(gidx), device=dev), gidx] = 0.0
    pds = pds_surrogate(pred, y, gidx, w, max_b=PDS_B)
    shape = topk_shape_loss(pred, y, gidx, max_b=PDS_B)
    if was_training:
        decoder.train()
    return pds, shape


@torch.no_grad()
def decoder_probe(batches=DECODER_PROBE_BATCHES, seed=SEED + 7000):
    was_training = decoder.training
    decoder.eval()
    r = np.random.default_rng(seed)
    vals, maes, corrs = [], [], []
    for _ in range(batches):
        z_gene, ctx, y, _, gidx = matcher_batch(
            va_rows, _va_map, VA_M_CTX, VA_M_P, VA_BY_CTX, r)
        q = frozen_matcher_query(z_gene, ctx)
        pred = decoder(q)
        pred[torch.arange(len(gidx), device=dev), gidx] = 0.0
        vals.append(hard_pds_batch(pred, y, gidx))
        maes.append(float((pred - y).abs().mean()))
        a = pred - pred.mean(1, keepdim=True)
        b = y - y.mean(1, keepdim=True)
        corrs.append(float(((a * b).sum(1) /
                            (a.norm(dim=1) * b.norm(dim=1) + 1e-9)).mean()))
    if was_training:
        decoder.train()
    return (float(np.mean(vals)), float(np.mean(maes)), float(np.mean(corrs)))


best_decoder_pds = -1.0
best_decoder_state = None

for step in range(1, DECODER_STEPS + 1):
    decoder.train()
    t = step / max(DECODER_STEPS - 1, 1)
    ramp = min(1.0, t / DECODER_RAMP)

    q, y, w, gidx = decoder_row_batch(
        tr_rows, _tr_map, DECODER_BATCH, decoder_rng)
    loss = decoder_pointwise_loss(q, y, w, gidx)

    pds_term, shape_term = decoder_context_terms()
    loss = loss + ramp * (DECODER_PDS_W * pds_term +
                          DECODER_SHAPE_W * shape_term)
    if not torch.isfinite(loss):
        raise FloatingPointError(f"decoder loss became non-finite at step {step}")

    dopt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
    dopt.step()

    if step % DECODER_LOG_EVERY == 0 or step == DECODER_STEPS:
        probe = decoder_probe(seed=SEED + 7000 + step)
        if probe[0] > best_decoder_pds:
            best_decoder_pds = probe[0]
            best_decoder_state = {k: v.detach().cpu().clone()
                                  for k, v in decoder.state_dict().items()}
        print(f"  decoder step {step:>4}  loss {float(loss.detach()):.4f}  "
              f"pdsL {float(pds_term.detach()):.3f}  "
              f"heldPDS {probe[0]:.4f}  MAE {probe[1]:.5f}  "
              f"corr {probe[2]:+.4f}  best {best_decoder_pds:.4f}")

if best_decoder_state is not None:
    decoder.load_state_dict(best_decoder_state)
    decoder.to(dev)
    print(f"  restored decoder state with held-out corpus PDS {best_decoder_pds:.4f}")

decoder_held_probe = decoder_probe(batches=DECODER_FINAL_PROBES,
                                   seed=SEED + 7900)
# The function above probes held-out rows; also make a training-row probe.
@torch.no_grad()
def decoder_train_probe(batches=DECODER_FINAL_PROBES, seed=SEED + 7901):
    decoder.eval()
    r = np.random.default_rng(seed)
    vals = []
    for _ in range(batches):
        z_gene, ctx, y, _, gidx = matcher_batch(
            tr_rows, _tr_map, TR_M_CTX, TR_M_P, TR_BY_CTX, r)
        q = frozen_matcher_query(z_gene, ctx)
        pred = decoder(q)
        pred[torch.arange(len(gidx), device=dev), gidx] = 0.0
        vals.append(hard_pds_batch(pred, y, gidx))
    return float(np.mean(vals))

train_decoder_pds = decoder_train_probe()
print(f"  decoder corpus train PDS {train_decoder_pds:.4f}  "
      f"held-out PDS {decoder_held_probe[0]:.4f}  "
      f"MAE {decoder_held_probe[1]:.5f}  corr {decoder_held_probe[2]:+.4f}")


@torch.no_grad()
def decoder_predict(pert_names, ctxvec, batch=128):
    decoder.eval()
    gene_ids = np.array([GPOS.get(p, -1) for p in pert_names], dtype=int)
    out = np.zeros((len(gene_ids), G), dtype=np.float32)
    for start in range(0, len(gene_ids), batch):
        gg = gene_ids[start:start + batch]
        ok = gg >= 0
        if not ok.any():
            continue
        ggv = gg[ok]
        gidx = torch.as_tensor(ggv, dtype=torch.long, device=dev)
        z_gene = ZG[gidx]
        ctx = ctxvec[None, :].repeat(len(gidx), 1)
        q = frozen_matcher_query(z_gene, ctx)
        pred = decoder(q)
        pred[torch.arange(len(gidx), device=dev), gidx] = 0.0
        out[start + np.where(ok)[0]] = pred.cpu().numpy()
    return out


H1_decode = decoder_predict(H1_pert, H1_ctx)
np.savez_compressed("/kaggle/working/matcher_decoder_h1_train.npz",
                    R=H1_decode.astype(np.float32), perts=np.array(H1_pert))
print(f"  wrote /kaggle/working/matcher_decoder_h1_train.npz {H1_decode.shape}")


## H1 low-rank decoder adapter

The corpus decoder generalizes across corpus genes but does not transfer its
original-space geometry to H1. This cell freezes the matcher and corpus decoder,
then learns only a low-rank H1 residual from the 150 legal H1 training
perturbations. Internal H1 folds hold out genes, not validation rows. The final
H1 adapter is used only if the constrained H1 rule produces a meaningful PDS
gain without MAE or DES regression.


In [ ]:

# ============================================================
# H1 LOW-RANK DECODER ADAPTER
#
# Corpus decoder: learns the general gene->response mapping.
# H1 adapter: learns only the corpus->H1 context shift.
# Internal folds hold out H1 TRAINING genes; validation truth remains closed.
# ============================================================

H1_ADAPTER_RANK = 16
H1_ADAPTER_STEPS = 250
H1_ADAPTER_BATCH = 48
H1_ADAPTER_LR = 3e-3
H1_ADAPTER_WD = 1e-4
H1_ADAPTER_FOLDS = 5
H1_ADAPTER_PDS_W = 0.25
H1_ADAPTER_SHAPE_W = 0.03
H1_ADAPTER_EVAL_EVERY = 25
H1_ADAPTER_MIN_CV_GAIN = 0.010


class H1LowRankAdapter(nn.Module):
    def __init__(self, d=128, rank=16, n_genes=G):
        super().__init__()
        self.down = nn.Linear(d, rank, bias=False)
        self.up = nn.Linear(rank, n_genes, bias=False)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)

    def forward(self, q):
        return self.up(self.down(q))


_H1_B_ALL = torch.as_tensor(H1_lfc - H1_lfc.mean(0, keepdims=True),
                            dtype=torch.float32, device=dev)
_H1_G_ALL = np.array([GPOS.get(p, -1) for p in H1_pert], dtype=int)
_H1_OK_ALL = np.where(_H1_G_ALL >= 0)[0]
_H1_W_ALL = torch.as_tensor(H1_n, dtype=torch.float32, device=dev)
_H1_W_ALL = _H1_W_ALL / _H1_W_ALL.mean().clamp_min(1e-9)


@torch.no_grad()
def _h1_query(sel):
    sel = np.asarray(sel, dtype=int)
    gi = _H1_G_ALL[sel]
    gidx = torch.as_tensor(gi, dtype=torch.long, device=dev)
    z_gene = ZG[gidx]
    ctx = H1_ctx[None, :].repeat(len(gidx), 1)
    return frozen_matcher_query(z_gene, ctx), gidx


def _h1_truth(sel, gidx):
    y = _H1_B_ALL[torch.as_tensor(sel, dtype=torch.long, device=dev)].clone()
    y[torch.arange(len(gidx), device=dev), gidx] = 0.0
    return y


def _adapter_pred(adapter, q, gidx):
    with torch.no_grad():
        base = decoder(q)
    pred = base + adapter(q)
    pred[torch.arange(len(gidx), device=dev), gidx] = 0.0
    return pred


def _adapter_loss(adapter, sel):
    q, gidx = _h1_query(sel)
    y = _h1_truth(sel, gidx)
    w = _H1_W_ALL[torch.as_tensor(sel, dtype=torch.long, device=dev)]
    pred = _adapter_pred(adapter, q, gidx)

    pn = pred / (pred.norm(dim=1, keepdim=True) + 1e-9)
    yn = y / (y.norm(dim=1, keepdim=True) + 1e-9)
    point = (w * (1.0 - (pn * yn).sum(1) +
                  0.5 * (pred - y).abs().mean(1))).mean()
    pds = pds_surrogate(pred, y, gidx, w, max_b=min(32, len(sel)))
    shape = topk_shape_loss(pred, y, gidx, max_b=min(32, len(sel)))
    return point + H1_ADAPTER_PDS_W * pds + H1_ADAPTER_SHAPE_W * shape


@torch.no_grad()
def _h1_eval(adapter, sel):
    adapter.eval()
    q, gidx = _h1_query(sel)
    y = _h1_truth(sel, gidx)
    pred = _adapter_pred(adapter, q, gidx)
    pds = hard_pds_batch(pred, y, gidx)
    mae = float((pred - y).abs().mean())
    a = pred - pred.mean(1, keepdim=True)
    b = y - y.mean(1, keepdim=True)
    corr = float(((a * b).sum(1) /
                  (a.norm(dim=1) * b.norm(dim=1) + 1e-9)).mean())
    return {"pds": pds, "mae": mae, "corr": corr}


class _IdentityAdapter(nn.Module):
    def forward(self, q):
        return torch.zeros(q.shape[0], G, device=q.device)


def _train_h1_adapter(train_sel, eval_sel=None, steps=H1_ADAPTER_STEPS,
                      seed=SEED + 8200, verbose=False):
    adapter = H1LowRankAdapter(MATCH_DIM, H1_ADAPTER_RANK, G).to(dev)
    opt = torch.optim.AdamW(adapter.parameters(), lr=H1_ADAPTER_LR,
                            weight_decay=H1_ADAPTER_WD)
    r = np.random.default_rng(seed)
    best = {"pds": -1.0, "mae": float("inf"), "corr": 0.0}
    best_state = None
    best_step = 0

    for step in range(1, steps + 1):
        adapter.train()
        take = r.choice(train_sel, min(H1_ADAPTER_BATCH, len(train_sel)),
                        replace=False)
        loss = _adapter_loss(adapter, take)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"H1 adapter loss non-finite at step {step}")
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        opt.step()

        if eval_sel is not None and (step % H1_ADAPTER_EVAL_EVERY == 0 or step == steps):
            ev = _h1_eval(adapter, eval_sel)
            if ev["pds"] > best["pds"]:
                best = ev
                best_step = step
                best_state = {k: v.detach().cpu().clone()
                              for k, v in adapter.state_dict().items()}
            if verbose:
                print(f"    adapter step {step:>3}  loss {float(loss.detach()):.4f}  "
                      f"foldPDS {ev['pds']:.4f}  MAE {ev['mae']:.5f}")

    if best_state is not None:
        adapter.load_state_dict(best_state)
        adapter.to(dev)
    return adapter, best, best_step


# Five internal H1 cross-gene folds. This uses only the 150 legal training rows.
_r = np.random.default_rng(SEED + 8100)
_h1_order = _H1_OK_ALL.copy()
_r.shuffle(_h1_order)
_h1_folds = np.array_split(_h1_order, H1_ADAPTER_FOLDS)
_base_adapter = _IdentityAdapter().to(dev)
_cv_rows = []

print("\n  H1 adapter internal cross-gene folds")
print(f"  {'fold':>5}{'basePDS':>9}{'adaptPDS':>10}{'dPDS':>8}"
      f"{'baseMAE':>10}{'adaptMAE':>10}{'dCorr':>9}")
_best_steps = []
for _fi, _eval_sel in enumerate(_h1_folds):
    _train_sel = np.setdiff1d(_H1_OK_ALL, _eval_sel)
    _base = _h1_eval(_base_adapter, _eval_sel)
    _ad, _best, _best_step = _train_h1_adapter(
        _train_sel, _eval_sel, seed=SEED + 8200 + _fi)
    _best_steps.append(_best_step)
    _cv_rows.append((_base, _best))
    print(f"  {_fi:>5}{_base['pds']:>9.4f}{_best['pds']:>10.4f}"
          f"{_best['pds']-_base['pds']:>8.4f}{_base['mae']:>10.5f}"
          f"{_best['mae']:>10.5f}{_best['corr']-_base['corr']:>9.4f}")

_base_cv = {k: float(np.mean([x[0][k] for x in _cv_rows])) for k in _cv_rows[0][0]}
_ad_cv = {k: float(np.mean([x[1][k] for x in _cv_rows])) for k in _cv_rows[0][1]}
_cv_gain = _ad_cv["pds"] - _base_cv["pds"]
print(f"\n  H1 adapter CV mean: base PDS {_base_cv['pds']:.4f}, "
      f"adapted PDS {_ad_cv['pds']:.4f}, gain {_cv_gain:+.4f}; "
      f"MAE {_base_cv['mae']:.5f} -> {_ad_cv['mae']:.5f}")

# Train final adapter on all 150 legal H1 training rows only if CV showed a
# meaningful cross-gene gain. Otherwise H1_adapt falls back to the corpus decoder.
H1_ADAPTER_USED = bool(_cv_gain >= H1_ADAPTER_MIN_CV_GAIN)
if H1_ADAPTER_USED:
    _final_steps = max(H1_ADAPTER_EVAL_EVERY,
                       int(np.median(_best_steps)))
    H1_adapter, _, _ = _train_h1_adapter(
        _H1_OK_ALL, eval_sel=None, steps=_final_steps, seed=SEED + 8300)
    print(f"  trained final H1 adapter on {len(_H1_OK_ALL)} training rows "
          f"for {_final_steps} steps")
else:
    H1_adapter = _IdentityAdapter().to(dev)
    print(f"  H1 adapter rejected by internal CV; using corpus decoder unchanged")


@torch.no_grad()
def adapter_predict(pert_names, ctxvec, batch=128):
    decoder.eval(); H1_adapter.eval()
    gene_ids = np.array([GPOS.get(p, -1) for p in pert_names], dtype=int)
    out = np.zeros((len(gene_ids), G), dtype=np.float32)
    for start in range(0, len(gene_ids), batch):
        gg = gene_ids[start:start + batch]
        ok = gg >= 0
        if not ok.any():
            continue
        ggv = gg[ok]
        gidx = torch.as_tensor(ggv, dtype=torch.long, device=dev)
        z_gene = ZG[gidx]
        ctx = ctxvec[None, :].repeat(len(gidx), 1)
        q = frozen_matcher_query(z_gene, ctx)
        pred = decoder(q) + H1_adapter(q)
        pred[torch.arange(len(gidx), device=dev), gidx] = 0.0
        out[start + np.where(ok)[0]] = pred.cpu().numpy()
    return out


H1_adapt = adapter_predict(H1_pert, H1_ctx)
np.savez_compressed("/kaggle/working/matcher_decoder_h1_adapted_train.npz",
                    R=H1_adapt.astype(np.float32), perts=np.array(H1_pert))
print(f"  wrote /kaggle/working/matcher_decoder_h1_adapted_train.npz "
      f"{H1_adapt.shape}  (adapter used: {H1_ADAPTER_USED})")


## 6a. SMOKE TEST — 60 seconds, before anything expensive

Three consecutive multi-hour runs died on the same parameter, each time for a
different reason, and each failure was only visible after the full sweep:

```
  run 1   coupling too weak     |f| ~ 0.003, tanh = the bias, no propagation
                                -> 100% of prediction variance in the injected
                                   gene, off-target spec 0.0708, score 24.09
  run 2   coupling = 3.0        tanh 99.4% SATURATED, gradient ~0
                                -> all 13 architectures scored acc ~ 0.000
  run 3   RMS normalisation     sqrt() singular at f = 0 (which happens when a
                                perturbed gene is gated dead)
                                -> nan from every config, 451s x 39 = 5 hours
```

Every one of those is detectable in under a minute on a tiny config. The rule
from here: **if the operator cannot fit TRAINING data, the sweep is
meaningless**, so check that first and abort loudly rather than producing a
confident table of noise.

The decisive assertion is `train accuracy moves off zero`. Held-out accuracy
can legitimately be ~0 (that would be a real negative result about transfer).
Train accuracy at 0 means the model is broken, not that the biology is hard.


In [ ]:
SMOKE = True

if SMOKE:
    print("  SMOKE TEST: depth 2, width 256, 60 epochs — checking the operator "
          "can learn at all\n")
    _t0 = time.time()
    set_holdout(SEED)
    build_models(2, 256, seed=SEED)
    _fail = []
    _acc0 = accuracy(tr_rows[:256], _tr_map[:256])
    _sats, _accs = [], []
    for _e in range(60):
        _si = rng.choice(len(tr_rows), min(256, len(tr_rows)), replace=False)
        _ramp = min(1.0, _e / 30.0)
        _pds_w = PDS_W * _ramp if USE_PDS_LOSS else 0.0
        _shape_w = SHAPE_W * _ramp if USE_SHAPE_LOSS else 0.0
        _loss = batch_loss(tr_rows[_si], idx_sel=_tr_map[_si],
                           pds_w=0.0, shape_w=0.0)
        _pds_ce = 0.0
        if _pds_w > 0 or _shape_w > 0:
            _pt, _st = context_pds_terms()
            if _pds_w > 0:
                _loss = _loss + _pds_w * _pt
                _pds_ce = float(_pt.detach())
            if _shape_w > 0:
                _loss = _loss + _shape_w * _st
        if not torch.isfinite(_loss):
            _fail.append(f"non-finite loss at smoke epoch {_e}")
            break
        opt.zero_grad(); _loss.backward()
        _nb = [n for n, q in list(plife.named_parameters()) + list(hyper.named_parameters())
               if q.grad is not None and not torch.isfinite(q.grad).all()]
        if _nb:
            _fail.append(f"non-finite gradient at smoke epoch {_e}: {_nb[:3]}")
            break
        torch.nn.utils.clip_grad_norm_(
            list(plife.parameters()) + list(hyper.parameters())
            + list(viab.parameters()), 1.0)
        opt.step()
        if _e % 10 == 0 or _e == 59:
            _a = accuracy(tr_rows[:256], _tr_map[:256])
            _sats.append(plife._sat); _accs.append(_a)
            print(f"    ep {_e:>3}  loss {float(_loss.detach()):.4f}  "
                  f"pdsL {_pds_ce:.3f}  train acc {_a:+.4f}  "
                  f"tanh sat {100*plife._sat:5.1f}%  "
                  f"drive0 {100*plife._drive_zero:4.1f}%  "
                  f"gain_f {float(plife.gain_f):.3f}")

    _amax = max(_accs) if _accs else 0.0
    _smax = max(_sats) if _sats else 0.0
    if _smax > 0.90:
        _fail.append(f"tanh {100*_smax:.1f}% saturated -- gradient is ~0, nothing "
                     f"can learn (this is exactly run 2)")
    if abs(_amax) < 0.02:
        _fail.append(f"train accuracy never left zero (best {_amax:+.4f}) -- the "
                     f"operator cannot fit its OWN TRAINING DATA, so any sweep "
                     f"below would be ranking noise (this is exactly run 2)")
    print()
    if _fail:
        for f_ in _fail:
            print(f"  SMOKE TEST FAILED: {f_}")
        raise SystemExit("Aborting before the sweep. Fix the operator first — "
                         "a 5-hour run cannot rescue a model that will not fit "
                         "60 epochs of training data.")
    _pr = min(64, len(tr_rows))
    _pm = lifecycle_probe(tr_rows[:_pr], _tr_map[:_pr])
    print(f"  smoke probe: hard PDS {_pm['pds']:.4f}  MAE {_pm['mae']:.5f}  "
          f"off-target corr {_pm['corr']:+.4f}")
    print(f"  SMOKE TEST PASSED in {time.time()-_t0:.0f}s  "
          f"(train acc reached {_amax:+.4f}, tanh sat peaked {100*_smax:.1f}%)")
    print(f"  the operator learns. proceeding to the sweep.\n")


## 6b. How wide and how deep? — swept, not guessed

Two facts make this worth measuring rather than arguing about:

1. **Depth is quadratically expensive here.** Every layer feeds every later
   layer, so a depth-D trunk has D(D+1)/2 connection matrices:

   ```
     depth  matrices     params      params / independent gene
         2         3    707,072                            93
         3         6  1,428,480                           188
         4        10  2,412,032                           318
         6        21  5,165,568                           681
   ```

   Against 7,585 unique genes, depth 3 is already ~188 parameters per
   independent example *before* pruning. Every deep model in this project has
   lost to closed-form retrieval on exactly this ratio.

2. **But depth is what the synaptogenesis mechanism eats.** `streamline()`
   scans `for mid in range(1, depth)` — it needs a layer with something above
   AND below it to find an n1→n2→n3 relay. Depth 3 gives it **2 relay slots**.
   Starving the mechanism and then concluding it does nothing would be the
   same mistake as the death rule that could never fire.

So the sweep varies both, at roughly matched parameter budgets, and picks on
HELD-OUT-DONOR accuracy — not train loss, which always prefers more capacity.

Note `T_STEPS` is the *simulation* depth and is separate. If the worry is
"not deep enough to model a cascade", that is the knob, and it costs no
parameters at all.

Set `SWEEP = []` to skip and use the DEPTH / HID constants as given.


In [ ]:
# Compute is not the constraint; SELECTION NOISE is. A single short run per
# config makes the grid a lottery -- taking the argmax picks the luckiest run,
# not the best architecture. So every config is trained REPEATS times, each with
# a different donor holdout AND a different init, and configs are compared on
# the MEAN with the spread reported. If the top configs overlap within one
# standard deviation, the smaller one wins: they are not distinguishable and the
# smaller model generalises on priors.
SWEEP = []      # OFF. Architecture is locked above; searching further was
                # ranking noise on a metric whose ceiling is 0.1411.
SWEEP_EPOCHS = EPOCHS        # rank configs at the length they will actually run
SWEEP_REPEATS = 3            # seeds x rotated donor holdouts
SWEEP_STEPS = [3, 6, 10, 16]  # T_STEPS -- simulation depth, costs no parameters


def sweep_eval(depth, hid, repeats=SWEEP_REPEATS, epochs=SWEEP_EPOCHS):
    """Mean +- sd of held-out-donor accuracy over `repeats` independent runs."""
    accs, losses = [], []
    for r in range(repeats):
        set_holdout(SEED + 100 * r)          # rotate WHICH donors are held out
        build_models(depth, hid, seed=SEED + r)
        try:
            vl, va = train_models(epochs, record=False, verbose=False)
        except FloatingPointError as e:
            print(f"      repeat {r} DIVERGED: {e}")
            continue
        if not (np.isfinite(vl) and np.isfinite(va)):
            print(f"      repeat {r} produced non-finite metrics; dropped")
            continue
        accs.append(va); losses.append(vl)
    if not accs:
        return float("nan"), float("nan"), float("nan")
    a = np.array(accs)
    return a.mean(), a.std(), float(np.mean(losses))


if SWEEP:
    print(f"  sweeping {len(SWEEP)} (depth,width) x {SWEEP_REPEATS} repeats "
          f"x {SWEEP_EPOCHS} epochs = {len(SWEEP)*SWEEP_REPEATS} runs")
    print(f"  ranking on held-out-DONOR accuracy, holdout rotated per repeat\n")
    print(f"  {'depth':>6}{'width':>7}{'params':>12}{'relays':>8}"
          f"{'acc mean':>11}{'sd':>8}{'loss':>9}{'secs':>7}")
    res = []
    for _d, _h in SWEEP:
        _t = time.time()
        _n = build_models(_d, _h)
        _m, _s, _l = sweep_eval(_d, _h)
        res.append((_m, _s, _n, _d, _h))
        print(f"  {_d:>6}{_h:>7}{_n:>12,}{max(_d-1,0):>8}"
              f"{_m:>11.4f}{_s:>8.4f}{_l:>9.4f}{time.time()-_t:>7.0f}")

    best = max(res)
    # prefer the SMALLEST config statistically indistinguishable from the best
    tied = [r for r in res if r[0] >= best[0] - best[1]]
    tied.sort(key=lambda r: r[2])
    _m, _s, _n, DEPTH, HID = tied[0]
    print(f"\n  best mean: depth={best[3]} width={best[4]} acc {best[0]:+.4f}"
          f" +- {best[1]:.4f}")
    if (DEPTH, HID) != (best[3], best[4]):
        print(f"  {len(tied)} configs within 1 sd of it; taking the smallest -> "
              f"depth={DEPTH} width={HID} ({_n:,} params, acc {_m:+.4f})")
    else:
        print(f"  selected depth={DEPTH}, width={HID}")

    print(f"\n  sweeping T_STEPS (simulation depth; zero extra parameters)")
    print(f"  {'T_STEPS':>9}{'acc mean':>11}{'sd':>8}{'loss':>9}{'secs':>7}")
    _rs = []
    for _ts in SWEEP_STEPS:
        _t = time.time()
        T_STEPS = _ts
        _m, _s, _l = sweep_eval(DEPTH, HID)
        _rs.append((_m, _s, _ts))
        print(f"  {_ts:>9}{_m:>11.4f}{_s:>8.4f}{_l:>9.4f}{time.time()-_t:>7.0f}")
    _b = max(_rs)
    _tied = sorted([r for r in _rs if r[0] >= _b[0] - _b[1]], key=lambda r: r[2])
    T_STEPS = _tied[0][2]
    print(f"\n  selected T_STEPS={T_STEPS} (acc {_tied[0][0]:+.4f} +- {_tied[0][1]:.4f})")

    set_holdout(SEED)            # restore the canonical holdout for the real run
    HIST = {k: [] for k in HIST}
    hist.clear()


## 7. Train the selected model


In [ ]:
print(f"  FINAL RUN: depth {DEPTH}, width {HID}, T_STEPS {T_STEPS}, "
      f"density {DENSITY}, {EPOCHS} epochs")
print(f"  holdout: {HOLDOUT_MODE} (cross-gene ceiling ~0.26)")
print(f"  target to beat on VCC: adaptive-dropout retrieval 40.00, "
      f"CIRCE reproduced 38.15, predict-ZERO 25.10\n")
set_holdout(SEED)
build_models(DEPTH, HID)
_vl, _va = train_models(EPOCHS)
print(f"\n  final held-out({HOLDOUT_MODE}) loss {_vl:.4f}  accuracy {_va:+.4f}"
      f"   [cross-gene oracle ceiling ~0.2647]")


## 7a. H1 as the dictatorial set

The rules are now fixed. H1 does not revise them — it constrains their output.
The emission scale is fitted on the 150 H1 training perturbations under
FOLD-OUT, against the REAL VCC score on random 50-subsets.

The previous version computed a least-squares gain (392.2) and then threw it
away on the next line by renormalising to `|B1|.mean()`. Now the scale is
selected by the metric it will be judged on.


In [ ]:
@torch.no_grad()
def simulate(pert_names, ctxvec):
    gi = torch.tensor([GPOS[p] for p in pert_names], device=dev)
    e = torch.cat([ZG[gi], ctxvec[None, :].repeat(len(gi), 1)], 1)
    sp, gm, dt, gn, tau = hyper(e)
    tau = tau if USE_DYNAMIC_SPARSITY else None
    return plife(injection(gi), sp, gm, dt, gn, ZG, viab.gate(ZG),
                 tau=tau).cpu().numpy()


hyper.eval(); plife.eval()
H1_sim = simulate(H1_pert, H1_ctx)
B1 = H1_lfc - H1_lfc.mean(0)
_tc = np.array([GPOS.get(p, -1) for p in H1_pert])


def _spec_off_target(A, Y, tcol):
    """Correlation EXCLUDING the perturbed gene -- that column is injected, not
    predicted, and PDS subtracts it. Including it read +0.3182 when the honest
    number was +0.0708."""
    a, b = A.copy(), Y.copy()
    for i, c in enumerate(tcol):
        if c >= 0:
            a[i, c] = 0; b[i, c] = 0
    a = a - a.mean(0, keepdims=True); b = b - b.mean(0, keepdims=True)
    a = a - a.mean(1, keepdims=True); b = b - b.mean(1, keepdims=True)
    return float(np.nanmean((a * b).sum(1) / np.maximum(
        np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1), 1e-30)))


print(f"  raw simulation vs H1, ALL genes        "
      f"{_spec_off_target(H1_sim, B1, [-1]*len(H1_pert)):+.4f}")
print(f"  raw simulation vs H1, OFF-TARGET only  "
      f"{_spec_off_target(H1_sim, B1, _tc):+.4f}   <- the honest number")

# SAVE THE TRAINING PREDICTIONS. Without them, any post-processing choice
# (rank-matching vs retrieval-projection vs blending, and their parameters)
# could only be selected by reading validation scores -- which is the one thing
# the challenge forbids. With them, the choice is made on the 150 H1 training
# perturbations and validation stays untouched.
np.savez_compressed("/kaggle/working/plife_h1_train.npz",
                    P=H1_sim.astype(np.float32), perts=np.array(H1_pert))
print(f"  wrote /kaggle/working/plife_h1_train.npz {H1_sim.shape} "
      f"(needed to select post-processing LEGALLY)")


## 7b. Predict the 50 validation perturbations and score — ONCE


In [ ]:
VB_DES, VB_PDS, VB_MAE = 0.1058, 0.5129, 0.0266


def vcc_pds(P_, T_, tcol):
    N = len(T_); D = np.empty((N, N))
    for i in range(N):
        D[i] = np.abs(P_[i][None, :] - T_).sum(1)
        if tcol[i] >= 0:
            D[i] -= np.abs(P_[i, tcol[i]] - T_[:, tcol[i]])
    r = np.argsort(np.argsort(D, 1), 1)[np.arange(N), np.arange(N)]
    return float(np.mean(1 - r / max(N - 1, 1)))


def vcc_des(sc, de):
    o = []
    for i, d in enumerate(de):
        d = np.atleast_1d(d).astype(int); k = len(d)
        if k:
            o.append(len(set(np.argpartition(-sc[i], k - 1)[:k].tolist())
                          & set(d.tolist())) / k)
    return float(np.mean(o))


def vcc_score(d, pd_, m):
    return (max(0, (d - VB_DES) / (1 - VB_DES)) + max(0, (pd_ - VB_PDS) / (1 - VB_PDS))
            + max(0, (VB_MAE - m) / VB_MAE)) / 3 * 100


med = np.median(H1_lfc, 0)
ds = np.load(os.path.join(INPUT_DIR, "vcc_de_sets.npz"), allow_pickle=True)
dm = {str(p): x for p, x in zip(ds["perts"], ds["de"])}
fr = np.zeros(G); nd = 0
deT = []
for p_ in H1_pert:
    x = np.atleast_1d(dm.get(p_, np.array([], int))).astype(int)
    deT.append(x)
    if len(x): fr[x] += 1; nd += 1
fr /= max(nd, 1)
wg = ((fr + 1e-3) / (fr.mean() + 1e-3)) ** 0.25


def _mask_targets(A, tcol):
    A = A.copy()
    for i, c in enumerate(tcol):
        if c >= 0:
            A[i, c] = 0.0
    return A


# The target channel is PDS-blind but MAE-visible. Fit its scalar separately on
# H1 training so it cannot set the scale of the off-target response.
_ok = _tc >= 0
_st = H1_sim[_ok, _tc[_ok]]
_yt = H1_lfc[_ok, _tc[_ok]]
_bt = med[_tc[_ok]]
TARGET_BETA = float(np.clip(((_st * (_yt - _bt)).sum() /
                             ((_st * _st).sum() + 1e-12)), -1.0, 2.0))
print(f"  target-channel beta fitted on H1 training: {TARGET_BETA:.4f}")


def _dress(SIM, alpha, tcol, target_beta=TARGET_BETA):
    """Add the fixed H1 median without letting the injected target set scale."""
    B_off = _mask_targets(B1, _tc)
    S_off = _mask_targets(SIM, tcol)
    scale = np.abs(B_off).mean() / (np.abs(S_off).mean() + 1e-9)
    SP = S_off * scale
    P_ = med[None, :] + alpha * SP * wg[None, :]

    rr = np.arange(len(SIM))
    ok = tcol >= 0
    P_[rr[ok], tcol[ok]] = (med[tcol[ok]] +
                            target_beta * SIM[rr[ok], tcol[ok]])

    # Freeze DES to the training-derived frequency prior. This guarantees the
    # PDS intervention cannot degrade DES through the model's magnitude ranking.
    der = np.repeat(fr[None, :], len(SIM), axis=0)
    return P_, der


def _train_metrics(SIM, alpha, beta, sel):
    Pt, dt_ = _dress(SIM, alpha, _tc, target_beta=beta)
    return {
        "mae": float(np.abs(Pt[sel] - H1_lfc[sel]).mean()),
        "pds": vcc_pds(Pt[sel], H1_lfc[sel], _tc[sel]),
        "des": vcc_des(dt_[sel], [deT[i] for i in sel]),
    }


# Constrained selection on H1 TRAINING. The safe alpha=0 candidate is the
# median-plus-DE-prior incumbent; a positive alpha must preserve MAE and DES and
# improve PDS. Do not argmax total score here: that permits side-metric regressions.
_r = np.random.default_rng(0)
_subsets = [_r.choice(len(H1_pert), min(50, len(H1_pert)), replace=False)
            for _ in range(12)]


def _mean_metrics(alpha, beta):
    vals = [_train_metrics(H1_sim, alpha, beta, s) for s in _subsets]
    return {k: float(np.mean([x[k] for x in vals])) for k in vals[0]}


safe = _mean_metrics(0.0, 0.0)
print(f"  H1-training safe incumbent: PDS {safe['pds']:.4f}  "
      f"MAE {safe['mae']:.5f}  DES {safe['des']:.4f}")
print(f"  {'alpha':>7}{'beta':>7}{'PDS':>9}{'MAE':>10}{'DES':>9}{'status':>10}")

cands = []
for _al in (0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8):
    _m = _mean_metrics(_al, TARGET_BETA)
    ok = (_m["mae"] <= safe["mae"] and
          _m["des"] >= safe["des"] - 1e-6 and
          (_al == 0.0 or _m["pds"] >= safe["pds"] + 0.010))
    cands.append((_m["pds"], -_m["mae"], _al, TARGET_BETA, _m, ok))
    print(f"  {_al:>7.2f}{TARGET_BETA:>7.3f}{_m['pds']:>9.4f}"
          f"{_m['mae']:>10.5f}{_m['des']:>9.4f}"
          f"{'ACCEPT' if ok else 'reject':>10}")

eligible = [x for x in cands if x[5]]
if eligible:
    _pds, _negmae, ALPHA, BETA, _m, _ = max(
        eligible, key=lambda x: (x[0], x[1], -x[2]))
else:
    ALPHA, BETA, _m = 0.0, 0.0, safe
print(f"\n  constrained H1 selection: alpha={ALPHA}, beta={BETA:.4f}, "
      f"PDS {_m['pds']:.4f}, MAE {_m['mae']:.5f}, DES {_m['des']:.4f}\n")


In [ ]:

# ============================================================
# CONSTRAINED H1-TRAINING SELECTION: plife vs retrieval vs decoder vs H1 adapter
#
# A candidate must produce a MEANINGFUL PDS gain, not a noise-level 0.0002.
# ============================================================

PLIFE_ALPHA, PLIFE_BETA, PLIFE_M = ALPHA, BETA, _m
MIN_PDS_GAIN = 0.010


def _mean_metrics_for(SIM, alpha, beta):
    vals = [_train_metrics(SIM, alpha, beta, s) for s in _subsets]
    return {k: float(np.mean([x[k] for x in vals])) for k in vals[0]}


def _select_constrained(SIM, label):
    print(f"\n  {label} constrained selection")
    print(f"  {'alpha':>7}{'beta':>7}{'PDS':>9}{'MAE':>10}{'DES':>9}{'status':>10}")
    cands = []
    for _al in (0.0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.25):
        _mm = _mean_metrics_for(SIM, _al, 0.0)
        ok = (_mm["mae"] <= safe["mae"] and
              _mm["des"] >= safe["des"] - 1e-6 and
              (_al == 0.0 or _mm["pds"] >= safe["pds"] + MIN_PDS_GAIN))
        cands.append((_mm["pds"], -_mm["mae"], _al, 0.0, _mm, ok))
        print(f"  {_al:>7.2f}{0.0:>7.3f}{_mm['pds']:>9.4f}"
              f"{_mm['mae']:>10.5f}{_mm['des']:>9.4f}"
              f"{'ACCEPT' if ok else 'reject':>10}")
    eligible = [x for x in cands if x[5] and x[2] > 0]
    if eligible:
        pds, negmae, al, be, mm, _ = max(
            eligible, key=lambda x: (x[0], x[1], -x[2]))
        return al, be, mm, True
    return 0.0, 0.0, safe, False


RETR_ALPHA, RETR_BETA, RETR_M, RETR_OK = _select_constrained(
    H1_match, "matcher retrieval")
DEC_ALPHA, DEC_BETA, DEC_M, DEC_OK = _select_constrained(
    H1_decode, "matcher decoder")
ADP_ALPHA, ADP_BETA, ADP_M, ADP_OK = _select_constrained(
    H1_adapt, "H1-adapted decoder")

finalists = [("plife", PLIFE_ALPHA, PLIFE_BETA, PLIFE_M)]
if RETR_OK:
    finalists.append(("retrieval", RETR_ALPHA, RETR_BETA, RETR_M))
if DEC_OK:
    finalists.append(("decoder", DEC_ALPHA, DEC_BETA, DEC_M))
if ADP_OK:
    finalists.append(("adapter", ADP_ALPHA, ADP_BETA, ADP_M))

FINAL_SOURCE, ALPHA, BETA, _m = max(
    finalists, key=lambda x: (x[3]["pds"], -x[3]["mae"]))

print("\n  LEGAL MODEL COMPARISON")
print(f"  plife:      PDS {PLIFE_M['pds']:.4f}  MAE {PLIFE_M['mae']:.5f}  "
      f"DES {PLIFE_M['des']:.4f}")
print(f"  retrieval:  PDS {RETR_M['pds']:.4f}  MAE {RETR_M['mae']:.5f}  "
      f"DES {RETR_M['des']:.4f}  accepted={RETR_OK}")
print(f"  decoder:    PDS {DEC_M['pds']:.4f}  MAE {DEC_M['mae']:.5f}  "
      f"DES {DEC_M['des']:.4f}  accepted={DEC_OK}")
print(f"  H1 adapter: PDS {ADP_M['pds']:.4f}  MAE {ADP_M['mae']:.5f}  "
      f"DES {ADP_M['des']:.4f}  accepted={ADP_OK}  cv={H1_ADAPTER_USED}")
print(f"\n  FINAL TRAINING-SELECTED SOURCE: {FINAL_SOURCE.upper()}  "
      f"alpha={ALPHA}, beta={BETA:.4f}, PDS {_m['pds']:.4f}, "
      f"MAE {_m['mae']:.5f}, DES {_m['des']:.4f}\n")


In [ ]:
if not SCORE_VALIDATION:
    print("SCORE_VALIDATION=False -- validation truth was not read.")
else:
    v = np.load(os.path.join(INPUT_DIR, "validation_truth.npz"), allow_pickle=True)
    TRUE = v["true"].astype(np.float32)
    vp = [str(x) for x in v["perts"]]
    de_val = v["de"]
    tcolV = np.array([GPOS.get(p, -1) for p in vp])
    if FINAL_SOURCE == "retrieval":
        V_sim = matcher_retrieve(vp, H1_ctx)
    elif FINAL_SOURCE == "decoder":
        V_sim = decoder_predict(vp, H1_ctx)
    elif FINAL_SOURCE == "adapter":
        V_sim = adapter_predict(vp, H1_ctx)
    else:
        V_sim = simulate(vp, H1_ctx)
    P, der = _dress(V_sim, ALPHA, tcolV, target_beta=BETA)

    def _line(nm, A, dd=None, note=""):
        m_ = float(np.abs(A - TRUE).mean())
        p_ = vcc_pds(A, TRUE, tcolV)
        d_ = vcc_des(np.abs(A) if dd is None else dd, de_val)
        print(f"  {nm:<38}{_spec_off_target(A, TRUE, tcolV):>9.4f}{m_:>9.4f}"
              f"{p_:>8.4f}{d_:>8.4f}{vcc_score(d_, p_, m_):>8.2f}  {note}")

    print(f"  {'':<38}{'SPEC*':>9}{'MAE':>9}{'PDS':>8}{'DES':>8}{'SCORE':>8}")
    _line("predict ZERO", np.zeros_like(TRUE))
    _line("training-median constant", np.repeat(med[None, :], len(vp), 0))
    _line(FINAL_SOURCE.upper() + " (selected)", P, der, "<- the model")
    _line("PERFECT", TRUE)
    print(f"\n  SPEC* excludes the perturbed gene (PDS does too)")
    print(f"  DES is frozen to the H1 DE-frequency prior for constrained PDS work")
    print(f"  reference: CIRCE reproduced 38.15 | adaptive-dropout retrieval 40.00")

    np.savez_compressed("/kaggle/working/plife_pred.npz",
                        P=P.astype(np.float32), de_score=der.astype(np.float32),
                        perts=np.array(vp))
    print(f"\n  WROTE /kaggle/working/plife_pred.npz  {P.shape}")


## 8. Diagnostics — learning, accuracy, and the surviving network

Three things worth reading in order:

1. **Learning curves.** Train vs held-out DONOR. If they diverge, the
   hypernetwork is memorising genetic backgrounds rather than learning rules,
   and nothing downstream will transfer to H1.
2. **Accuracy curves.** Mean per-perturbation correlation between predicted
   and true delta — the regression analogue of accuracy. Loss can fall while
   this stays flat if the model is only fitting magnitude.
3. **The network map.** Every unit, its outgoing weight strength, and the
   dead ones in black. This is where the lifecycle either shows structure or
   reveals it never fired.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

H = HIST
fig = plt.figure(figsize=(16, 13))
gs = fig.add_gridspec(3, 3, hspace=0.38, wspace=0.28)

# ---------- 1. learning curves ----------
ax = fig.add_subplot(gs[0, 0])
ax.plot(H["ep"], H["train"], label="train", lw=2)
ax.plot(H["ep"], H["val"], label="held-out DONOR", lw=2)
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("learning curves")
ax.legend(); ax.grid(alpha=.3)

# ---------- 2. accuracy curves ----------
ax = fig.add_subplot(gs[0, 1])
ax.plot(H["ep"], H["acc_tr"], label="train", lw=2)
ax.plot(H["ep"], H["acc_va"], label="held-out DONOR", lw=2)
ax.axhline(0, color="grey", lw=.8)
ax.set_xlabel("epoch"); ax.set_ylabel("mean corr(pred, true)")
ax.set_title("accuracy — per-perturbation correlation")
ax.legend(); ax.grid(alpha=.3)

# ---------- 3. population dynamics ----------
ax = fig.add_subplot(gs[0, 2])
ax.plot(H["ep"], H["density"], lw=2, color="tab:blue", label="synapse density")
ax.fill_between(H["ep"], H["pd_lo"], H["pd_hi"], alpha=.25, color="tab:orange",
                label="p_drop range")
ax.plot(H["ep"], H["coupling"], ls=":", color="tab:orange", label="coupling")
ax.set_xlabel("epoch"); ax.set_ylabel("fraction"); ax.set_title("lifecycle")
ax2 = ax.twinx()
ax2.plot(H["ep"], H["units_dead"], color="crimson", lw=2, label="units dead")
ax2.plot(H["ep"], H["skips"], color="tab:green", lw=2, label="skips grown")
ax2.set_ylabel("count")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=7, loc="center right"); ax.grid(alpha=.3)

# ---------- 4-6. the neuron map, one panel per layer ----------
@torch.no_grad()
def unit_out_strength(l):
    w = torch.zeros(hyper.hid, device=dev)
    for i, (a, b) in enumerate(hyper.pairs):
        if a == l:
            w += (hyper.lin[i].weight * hyper.masks[i]).abs().sum(0)
    if l == hyper.depth:
        w += hyper.head.weight.abs().sum(0)
    return w.cpu().numpy()

cmap = plt.cm.viridis.copy(); cmap.set_bad("black")
for l in range(1, hyper.depth + 1):
    ax = fig.add_subplot(gs[1, l - 1])
    v = unit_out_strength(l)
    alive = hyper.unit_alive[l].cpu().numpy() > 0
    vm = np.where(alive, v, np.nan)
    cols = 32
    rows_ = int(np.ceil(len(vm) / cols))
    pad = np.full(rows_ * cols - len(vm), np.nan)
    im = ax.imshow(np.concatenate([vm, pad]).reshape(rows_, cols),
                   cmap=cmap, aspect="auto")
    nd = int((~alive).sum())
    ax.set_title(f"layer {l}: {int(alive.sum())} alive, {nd} dead (black)\n"
                 f"colour = outgoing weight strength", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=.046)

# ---------- 7. every unit in one strip, labelled BY LAYER ----------
ax = fig.add_subplot(gs[2, 0])
strip, lay, alv = [], [], []
for l in range(1, hyper.depth + 1):
    v = unit_out_strength(l)
    a = hyper.unit_alive[l].cpu().numpy() > 0
    strip.append(v); lay.append(np.full(len(v), l)); alv.append(a)
strip = np.concatenate(strip); lay = np.concatenate(lay); alv = np.concatenate(alv)
# one row per layer so layer membership is unambiguous
M = np.full((hyper.depth, hyper.hid), np.nan)
for l in range(1, hyper.depth + 1):
    v = strip[lay == l].copy()
    v[~alv[lay == l]] = np.nan                        # dead -> black
    M[l - 1] = v
im = ax.imshow(M, cmap=cmap, aspect="auto", interpolation="nearest")
ax.set_yticks(range(hyper.depth))
ax.set_yticklabels([f"layer {l}\n{int(alv[lay==l].sum())} alive / "
                    f"{int((~alv[lay==l]).sum())} dead"
                    for l in range(1, hyper.depth + 1)], fontsize=8)
ax.set_xlabel("unit index within layer")
ax.set_title("every unit by LAYER — colour = outgoing strength, black = dead",
             fontsize=9)
fig.colorbar(im, ax=ax, fraction=.046)

# ---------- 8. the mechanism: connectivity vs drop probability ----------
ax = fig.add_subplot(gs[2, 1])
for l in range(1, hyper.depth + 1):
    inc = torch.zeros(hyper.hid, device=dev)
    for i, (a, b) in enumerate(hyper.pairs):
        if b == l:
            inc += hyper.masks[i].sum(1)
    ax.scatter(inc.cpu().numpy(), hyper._pdrop[l].cpu().numpy(), s=6, alpha=.45,
               label=f"layer {l}")
ax.set_xlabel("surviving in-edges"); ax.set_ylabel("p(drop)")
ax.set_title("connectivity-weighted dropout\n(down-sloping = mechanism is live)",
             fontsize=9)
ax.legend(fontsize=8); ax.grid(alpha=.3)

# ---------- 9. synapse map, including grown cross-layer skips ----------
ax = fig.add_subplot(gs[2, 2])
sk = hyper._idx.get((0, hyper.depth))
if sk is not None:
    M = hyper.masks[sk].cpu().numpy()
    ax.imshow(M[:128, :128], cmap="Greys", aspect="auto", interpolation="nearest")
    ax.set_title(f"L0 -> L{hyper.depth} skip matrix (top-left 128x128)\n"
                 f"{int(M.sum())} live synapses, {100*M.mean():.1f}% dense",
                 fontsize=9)
else:
    ax.text(.5, .5, "no cross-layer skip matrix", ha="center",
            transform=ax.transAxes)
ax.set_xlabel("source unit"); ax.set_ylabel("target unit")

# per-layer census, printed so the numbers are exact and not eyeballed
print("\n  per-layer census")
print(f"  {'layer':<8}{'alive':>8}{'dead':>7}{'density':>10}"
      f"{'mean out-strength (alive)':>28}")
for l in range(1, hyper.depth + 1):
    v = unit_out_strength(l); a = hyper.unit_alive[l].cpu().numpy() > 0
    d_in = sum(float(hyper.masks[i].mean()) for i, (x, y) in enumerate(hyper.pairs)
               if y == l)
    n_in = sum(1 for (x, y) in hyper.pairs if y == l)
    print(f"  {l:<8}{int(a.sum()):>8}{int((~a).sum()):>7}"
          f"{d_in / max(n_in, 1):>10.3f}{v[a].mean() if a.any() else 0:>28.4f}")

fig.suptitle("plife-VCC — training, accuracy, and the surviving network",
             fontsize=13, y=.995)
plt.savefig("/kaggle/working/plife_diagnostics.png", dpi=140, bbox_inches="tight")
plt.show()

# ---------- the gene population ----------
fig3, axw = plt.subplots(1, hyper.depth, figsize=(5 * hyper.depth, 3.6))
axw = np.atleast_1d(axw)
for l in range(1, hyper.depth + 1):
    v = unit_out_strength(l); a = hyper.unit_alive[l].cpu().numpy() > 0
    bins = np.linspace(0, max(v.max(), 1e-9), 40)
    axw[l - 1].hist(v[a], bins=bins, alpha=.8, color="tab:blue",
                    label=f"alive {int(a.sum())}")
    if (~a).any():
        axw[l - 1].hist(v[~a], bins=bins, alpha=.8, color="crimson",
                        label=f"dead {int((~a).sum())}")
    else:
        axw[l - 1].text(.5, .6, "no deaths in this layer", ha="center",
                        transform=axw[l - 1].transAxes, fontsize=9, color="crimson")
    axw[l - 1].set_title(f"layer {l}: outgoing weight strength", fontsize=10)
    axw[l - 1].set_xlabel("sum |outgoing weight|"); axw[l - 1].legend(fontsize=8)
axw[0].set_ylabel("units")
plt.tight_layout()
plt.savefig("/kaggle/working/plife_units_by_layer.png", dpi=140, bbox_inches="tight")
plt.show()

fig2, ax = plt.subplots(1, 3, figsize=(16, 4))
g_alive = viab.alive.cpu().numpy() > 0
gate = viab.gate(ZG).detach().cpu().numpy()
ax[0].plot(H["ep"], H["genes_dead"], color="crimson", lw=2)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("genes dead")
ax[0].set_title(f"gene death ({int((~g_alive).sum())} of {G} dead)"); ax[0].grid(alpha=.3)
ax[1].hist(np.log10(gate[g_alive] + 1e-6), bins=60, color="tab:blue")
ax[1].set_xlabel("log10 participation gate"); ax[1].set_ylabel("genes")
ax[1].set_title("competitive gene gate (mean 1 by construction)")
ax[2].scatter(np.log10(ACT.cpu().numpy() + 1e-6), np.log10(gate + 1e-6),
              s=3, alpha=.25,
              c=np.where(g_alive, "tab:blue", "crimson"))
ax[2].set_xlabel("log10 corpus activity"); ax[2].set_ylabel("log10 gate")
ax[2].set_title("gate vs measured activity (red = dead)")
plt.tight_layout()
plt.savefig("/kaggle/working/plife_genes.png", dpi=140, bbox_inches="tight")
plt.show()


## 9. The network itself — neurons as dots, synapses as lines

Units are placed by LAYER (x) and index (y). Dot size = outgoing strength,
colour = alive/dead. Lines are surviving synapses: **red = negative weight,
blue = positive**, opacity by |w|. Long arcs that skip a layer are the
cross-layer connections grown by `streamline()`.

Weights are unconstrained in sign; the fraction negative is printed, so if a
layer has collapsed to one sign that is visible rather than assumed.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

MAX_NODES_PER_LAYER = 90      # subsample for legibility; totals printed below
MAX_EDGES = 2500              # strongest surviving synapses

L = hyper.depth
pos, keep_idx, alive_all = {}, {}, {}
for l in range(1, L + 1):
    st = unit_out_strength(l)
    al = hyper.unit_alive[l].cpu().numpy() > 0
    alive_all[l] = al
    order = np.argsort(-st)
    sel = np.sort(order[:MAX_NODES_PER_LAYER])
    keep_idx[l] = sel
    y = np.linspace(0, 1, len(sel))
    for r, u in enumerate(sel):
        pos[(l, int(u))] = (float(l), float(y[r]))

fig, ax = plt.subplots(figsize=(13, 9))

segs, cols, lws = [], [], []
for i, (a, b) in enumerate(hyper.pairs):
    if a == 0:
        continue                                   # input layer has no dots here
    W = (hyper.lin[i].weight * hyper.masks[i]).detach().cpu().numpy()   # (dst, src)
    sa, sb = keep_idx.get(a), keep_idx.get(b)
    if sa is None or sb is None:
        continue
    sub = W[np.ix_(sb, sa)]
    if sub.size == 0:
        continue
    thr = np.quantile(np.abs(sub[sub != 0]), 0.90) if (sub != 0).any() else 0
    for r_i, dst in enumerate(sb):
        for c_i, src in enumerate(sa):
            w = sub[r_i, c_i]
            if w == 0 or abs(w) < thr:
                continue
            p0, p1 = pos[(a, int(src))], pos[(b, int(dst))]
            segs.append([p0, p1])
            cols.append((0.85, 0.15, 0.15) if w < 0 else (0.15, 0.35, 0.85))
            lws.append(abs(w))
if segs:
    o = np.argsort(-np.array(lws))[:MAX_EDGES]
    segs = [segs[k] for k in o]; cols = [cols[k] for k in o]
    lw = np.array([lws[k] for k in o]); lw = 0.2 + 1.8 * lw / (lw.max() + 1e-9)
    al = 0.05 + 0.35 * (lw - lw.min()) / (lw.max() - lw.min() + 1e-9)
    ax.add_collection(LineCollection(segs, colors=[(*c, a_) for c, a_ in zip(cols, al)],
                                     linewidths=lw, zorder=1))

for l in range(1, L + 1):
    st = unit_out_strength(l); al = alive_all[l]
    xs = [pos[(l, int(u))][0] for u in keep_idx[l]]
    ys = [pos[(l, int(u))][1] for u in keep_idx[l]]
    sz = st[keep_idx[l]]; sz = 12 + 180 * sz / (sz.max() + 1e-9)
    live = al[keep_idx[l]]
    ax.scatter(np.array(xs)[live], np.array(ys)[live], s=sz[live],
               c="#1f77b4", edgecolors="white", linewidths=.6, zorder=3,
               label="alive" if l == 1 else None)
    if (~live).any():
        ax.scatter(np.array(xs)[~live], np.array(ys)[~live], s=sz[~live],
                   c="black", marker="x", zorder=3,
                   label="dead" if l == 1 else None)

for l in range(1, L + 1):
    al = alive_all[l]
    neg = []
    for i, (a, b) in enumerate(hyper.pairs):
        if a == l:
            W = (hyper.lin[i].weight * hyper.masks[i]).detach().cpu().numpy()
            nz = W[W != 0]
            if nz.size: neg.append((nz < 0).mean())
    fneg = float(np.mean(neg)) if neg else 0.0
    ax.text(l, -0.07, f"layer {l}\n{int(al.sum())} alive / {int((~al).sum())} dead\n"
                      f"{100*fneg:.0f}% neg weights", ha="center", fontsize=9)

ax.set_xlim(0.5, L + 0.5); ax.set_ylim(-0.14, 1.05)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"plife hypernetwork — {MAX_NODES_PER_LAYER} strongest units per layer, "
             f"top {MAX_EDGES} synapses\nblue = positive weight, red = negative, "
             f"x = dead unit", fontsize=11)
ax.legend(loc="upper right", fontsize=9)
for sp in ax.spines.values():
    sp.set_visible(False)
plt.tight_layout()
plt.savefig("/kaggle/working/plife_network_map.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"  drawn: {len(segs):,} synapses over {sum(len(k) for k in keep_idx.values())} "
      f"of {L*hyper.hid:,} units")
print(f"  wrote /kaggle/working/plife_network_map.png")
